# 🔗 18 · Gaussian graphical models: networks from correlations
### Undirected graphs, the precision matrix, partial correlations and the graphical lasso — and a signalling network learned from single cells

**BiologyToML** · Level 1 · 02-620 core · Part D — Probabilistic graphical models · Module 18 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/18_gaussian_graphical_models.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | Modules 02 (Gaussian distribution), 03 (vectors, matrices, inverse), 06 (linear regression), 07 (lasso, coordinate descent), 15–16 (Bayesian networks, Markov blankets) |
| **Estimated time** | 160 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Gaussian graphical models: why undirected models for gene networks, Markov random fields and clique potentials, cliques, the pairwise, local and global Markov properties, discrete versus Gaussian MRFs; the multivariate normal distribution and the precision matrix ($\Theta_{ij}\neq0$ iff an edge); precision versus covariance; inference (marginals, conditionals, the connection to linear regression); maximum-likelihood learning ($\hat\Theta=S^{-1}$); sparse networks with the lasso penalty and the graphical lasso (block coordinate descent); efficient solvers; why Gaussianity makes undirected models tractable |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every data set in this notebook is **SIMULATED** by code. The single-cell signalling data of Section 10 come
> from the toolkit's `simulate_signaling_cytometry`, shaped like the flow-cytometry study of Sachs et al. (2005) — 11
> signalling molecules in 2,603 unperturbed cells — and generated from a linear-Gaussian model on the 20-edge consensus
> network, so the true network is known and we can score what a method recovers. The 8-node ring and the 5 × 5 example
> of the 02-620 lecture are typed in as precision matrices; all other networks are written down in the code. A learned
> network edge is a statistical dependence (a partial correlation), not proof of a biochemical interaction.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why undirected graphical models (Markov random fields) suit molecular networks, and **find** the cliques of a graph.
2. **State** the local, pairwise and global Markov properties of an undirected graph and **check** them by graph separation.
3. **Compute** with the multivariate Gaussian: density, sampling, marginal and conditional distributions.
4. **Explain** why the zeros of the precision matrix $\Theta=\Sigma^{-1}$ — not of the covariance matrix — are the missing edges, and **compute** partial correlations $-\Theta_{ij}/\sqrt{\Theta_{ii}\Theta_{jj}}$.
5. **Derive** that the distribution of one variable given all others is a linear regression with coefficients $-\Theta_{ij}/\Theta_{ii}$ and residual variance $1/\Theta_{ii}$, and **verify** it by regression on simulated data.
6. **Derive** the maximum-likelihood estimate $\hat\Theta=S^{-1}$ and **explain** why it is dense, noisy, and impossible when $N\le K$.
7. **Implement** the graphical lasso and neighbourhood selection from scratch and **evaluate** network recovery against a known graph.
8. **Learn** a network from SIMULATED single-cell signalling data whose true graph is known, **explain** why the true undirected graph of a linear-Gaussian Bayesian network is its *moral graph*, and **show** what a partial-correlation edge does *not* tell you (directions, unmeasured molecules).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · Why undirected models? | direction, feedback loops, Markov random fields | 8 min |
| 2 · Cliques and graph separation | brute-force cliques; local, pairwise, global Markov properties | 10 min |
| 3 · Discrete versus Gaussian MRFs | the normalising constant: $2^K$ terms versus one determinant | 7 min |
| 4 · The multivariate Gaussian | density, contours, sampling, the precision matrix | 15 min |
| 5 · The precision matrix is the graph | an 8-node ring and a 5 × 5 example; partial correlation | 18 min |
| 6 · Markov properties, verified numerically | exam marks; conditional covariances, graph separation, residuals | 16 min |
| 7 · Inference: conditionals are regressions | marginals, conditionals, $-\Theta_{ij}/\Theta_{ii}$ | 18 min |
| 8 · Learning by maximum likelihood | $\hat\Theta=S^{-1}$ and why it is not enough | 12 min |
| 9 · Sparsity: the graphical lasso | lasso, block coordinate descent, neighbourhood selection | 18 min |
| 10 · A T-cell signalling network (SIMULATED) | the moral graph as ground truth; two learners; hidden molecules; stability | 20 min |
| 11 · Summary, checkpoint, discussion | why Gaussian makes undirected models efficient; self-test | 10 min |

## 🧫 The biological question

Inside a T cell, a signal arriving at the surface is relayed by a cascade of proteins that add phosphate groups to one
another: RAF phosphorylates MEK, MEK phosphorylates ERK, and so on. Sachs and colleagues (2005) used **multiparameter
flow cytometry** to measure the phosphorylation of 11 signalling proteins and phospholipids *in each of thousands of
individual cells*. Every cell is one sample of a vector of 11 numbers. Cells differ — by chance some have more active
RAF, and then they also tend to have more active MEK — and this cell-to-cell variation carries information about who
talks to whom.

The question of this module: **from such measurements, which pairs of molecules are *directly* connected, and which are
only correlated because they share partners?** RAF and ERK are correlated, but is that because RAF acts on ERK, or only
through MEK? A Bayesian network (Modules 15–17) answers with arrows, but arrows are hard to learn from observational data
and cannot form feedback loops. A **Gaussian graphical model (GGM)** answers with undirected edges, and its answer has a
beautiful algebraic form: *the missing edges are the zeros of the inverse covariance matrix*. The same model is used for
gene networks learned from the expression profiles of hundreds of individuals — a refinement of the popular
co-expression networks, which link genes by their *marginal* correlation — and for microbiome association networks
and brain connectivity maps.

We will build every piece from scratch, check every identity numerically, and finally learn a signalling network from
single-cell data shaped like the Sachs study — and be honest about what such an edge does and does not mean.

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: every number in this notebook is generated
> by code. The signalling data of Section 10 come from the toolkit's `simulate_signaling_cytometry`, which imitates
> the Sachs *et al.* (2005) design (11 molecules, 2,603 unperturbed cells, the same molecule names) and generates the
> cells from a linear-Gaussian model on the 20-edge *consensus network* of that study. With real data the consensus
> network is only a summary of the literature; here it is the **true** network, so we can check exactly which edges a
> method found, which it missed and which it invented. None of the numbers below are the paper's numbers.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.covariance import graphical_lasso
from sklearn.linear_model import Lasso

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
They generate the SIMULATED single-cell signalling data (shaped like Sachs et al. 2005) from the 20-edge consensus network, draw small graphs with Matplotlib only, z-score columns and score recovered edges with precision and recall. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SACHS_PROTEINS = ["raf", "mek", "plc", "pip2", "pip3", "erk", "akt", "pka", "pkc", "p38", "jnk"]


SACHS_CONSENSUS_EDGES = [
    ("pka", "raf"), ("pkc", "raf"), ("raf", "mek"), ("pka", "mek"), ("pkc", "mek"),
    ("pip3", "plc"), ("plc", "pip2"), ("pip3", "pip2"), ("mek", "erk"), ("pka", "erk"),
    ("pip3", "akt"), ("erk", "akt"), ("pka", "akt"), ("pkc", "pka"), ("plc", "pkc"),
    ("pip2", "pkc"), ("pka", "p38"), ("pkc", "p38"), ("pka", "jnk"), ("pkc", "jnk"),
]


def simulate_signaling_cytometry(kind: str = "continuous", observational_only: bool = True,
                                 seed: int = 2005) -> dict:
    """Synthetic single-cell phospho-protein data shaped like the Sachs et al. (2005) signalling study.

    11 signalling proteins of human T cells, generated by a linear-Gaussian structural model on
    the 20-edge consensus network ``SACHS_CONSENSUS_EDGES`` (log intensities).  Besides
    unperturbed cells ("none"), some cells receive interventions that set one protein high
    (activator) or low (inhibitor): pkc (both), akt, pip2 and mek (inhibitors), pka (activator).
    ``kind='continuous'``: 7,466 cells; ``'discrete'``: 5,400 cells, each protein cut into
    low/medium/high (0/1/2) at the tertiles of the unperturbed cells.  The condition sizes follow
    the common benchmark encoding of the original data, in which the 2,603 "none" cells are the
    two general-stimulation conditions (853 + 902 cells) plus the 848 PI3K-inhibitor cells.  With
    ``observational_only=True`` only unperturbed cells are returned.  The two kinds are drawn
    with different seeds, so their edge strengths differ.

    Returns a dict with ``data`` (DataFrame, 11 proteins), ``intervention``, ``edges`` (the true
    network), ``weights`` (true edge strengths) and ``source``.
    """
    rng = np.random.default_rng(seed if kind == "continuous" else seed + 1)
    order = ["pip3", "plc", "pip2", "pkc", "pka", "raf", "mek", "erk", "akt", "p38", "jnk"]   # topological
    sign = {("pip3", "pip2"): -1, ("plc", "pip2"): -1}                   # PIP2 is consumed
    weights = {e: sign.get(e, 1) * rng.uniform(0.35, 0.85) for e in SACHS_CONSENSUS_EDGES}
    base = dict(zip(order, rng.uniform(2.5, 5.0, len(order))))
    noise = dict(zip(order, rng.uniform(0.45, 0.8, len(order))))
    sizes = ({"none": 2603, "pkc": 1636, "akt": 911, "pip2": 810, "mek": 799, "pka": 707} if kind == "continuous"
             else {"none": 1800, "pkc": 1200, "pka": 600, "akt": 600, "pip2": 600, "mek": 600})
    frames, labels = [], []
    for target, n in sizes.items():
        dev = {}
        for v in order:
            if v == target:                                              # intervention: do(v = high/low)
                direction = (np.where(rng.random(n) < 0.5, 1.0, -1.0) if v == "pkc"
                             else np.full(n, 1.0 if v == "pka" else -1.0))
                dev[v] = 1.6 * direction + rng.normal(0, 0.3, n)
            else:
                dev[v] = sum(w * dev[p] for (p, c), w in weights.items() if c == v) + rng.normal(0, noise[v], n)
        frames.append(pd.DataFrame({v: np.round(base[v] + dev[v], 4) for v in SACHS_PROTEINS}))
        labels += [target] * n
    data = pd.concat(frames, ignore_index=True)
    ivn = np.array(labels, dtype=object)
    if kind == "discrete":
        ref = data[ivn == "none"]
        for v in SACHS_PROTEINS:
            cuts = np.quantile(ref[v], [1 / 3, 2 / 3])
            data[v] = np.searchsorted(cuts, data[v].to_numpy()).astype(int)
    if observational_only:
        keep = ivn == "none"
        data, ivn = data[keep].reset_index(drop=True), ivn[keep]
    return dict(data=data, intervention=ivn, edges=list(SACHS_CONSENSUS_EDGES),
                weights=pd.Series({f"{p}→{c}": w for (p, c), w in weights.items()}),
                source="SIMULATED signalling data (shaped like Sachs et al. 2005)")


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_signaling_cytometry', 'standardize', 'confusion_counts', 'precision_recall_f1', 'draw_graph']))

---
## 1 · Why undirected models for molecular networks?

> 📍 **Where we are.** In Modules 15–17 you represented a joint distribution with a **directed** acyclic graph — a
> Bayesian network — and learned one from discrete signalling data simulated like the Sachs study. This section explains what goes wrong with arrows for
> regulatory networks and introduces the undirected alternative that this whole module is about.

Picture a gene regulatory network drawn as a Bayesian network (left panel of the next figure): a few **transcription
factors** (TFs) point to the **target genes** they regulate, and the targets point to **downstream genes**. *What is
the key challenge in learning such a model from data?* Two answers:

1. **Direction.** From observational data alone many edge directions are not identifiable (Module 17: $X\to Y\to Z$,
   $X\leftarrow Y\leftarrow Z$ and $X\leftarrow Y\rightarrow Z$ encode exactly the same independencies). Often we only
   know *that* two genes are linked, not *who regulates whom*.
2. **Cycles.** Biology is full of **feedback loops** — for example, active ERK phosphorylates RAF and dampens the very
   signal that activated it. A Bayesian network must be acyclic, so it cannot represent such a loop directly.

An **undirected graphical model**, or **Markov random field (MRF)**, drops the arrows: an edge only says
"these two variables interact directly". The two families side by side:

| | Bayesian network | Markov random field |
|---|---|---|
| graph | directed edges, no directed cycles | undirected edges, loops allowed |
| distribution | product of "local" conditional distributions, one per node | product of "clique potential functions", normalised so that the total probability is 1 |

The MRF distribution is no longer a product of conditionals $P(\text{child}\mid\text{parents})$; it is a normalised
product of non-negative **clique potentials** $\psi_C$:

$$p(x_1,\dots,x_K) \;=\; \frac{1}{Z}\prod_{C\,\in\,\text{cliques}} \psi_C(\mathbf x_C),
\qquad Z=\sum_{\text{all }\mathbf x}\;\prod_{C}\psi_C(\mathbf x_C).$$

*In words:* every group of directly connected variables (a clique $C$, defined in Section 2) gets a "compatibility
score" $\psi_C$ that is large for value combinations that like to occur together; multiplying all scores and dividing
by their grand total $Z$ turns them into probabilities that sum (or, for continuous variables, integrate) to 1.

🧮 **Math refresher — $\prod$ and normalising constants.** $\prod_{C}$ means "multiply over all $C$", just as $\sum$
means "add up". A **potential** is any non-negative function; unlike a conditional probability it does *not* have to sum
to one. The **normalising constant** $Z$ (also called the *partition function*) is the sum of the unnormalised product
over *every possible configuration* of all $K$ variables (an integral for continuous variables). Keep an eye on $Z$:
computing it is what makes discrete MRFs hard (Section 3) and Gaussian ones easy.

In [ ]:
# ── 1.1 A regulatory network as a Bayesian network, a feedback loop, and an undirected graph ──
GRN_POS = {"TF1": (0.15, 0.85), "TF2": (0.5, 0.85), "TF3": (0.85, 0.85),
           "t1": (0.05, 0.5), "t2": (0.35, 0.5), "t3": (0.65, 0.5), "t4": (0.95, 0.5),
           "d1": (0.2, 0.12), "d2": (0.5, 0.12), "d3": (0.8, 0.12)}
GRN_EDGES = [("TF1", "t1"), ("TF1", "t2"), ("TF2", "t2"), ("TF2", "t3"), ("TF3", "t3"), ("TF3", "t4"),
             ("t1", "d1"), ("t2", "d1"), ("t2", "d2"), ("t3", "d2"), ("t4", "d3"), ("t3", "d3")]
LOOP_POS = {"RAF": (0.15, 0.75), "MEK": (0.85, 0.75), "ERK": (0.5, 0.15)}
G6_NODES = [1, 2, 3, 4, 5, 6]
G6_EDGES = [(1, 2), (1, 5), (2, 3), (2, 5), (3, 4), (4, 5), (4, 6)]          # the six-variable example graph
G6_POS = {1: (0.95, 0.6), 2: (0.62, 0.8), 3: (0.25, 0.78), 4: (0.22, 0.38), 5: (0.6, 0.45), 6: (0.0, 0.02)}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.9))
draw_graph(GRN_EDGES, GRN_POS, directed=True, ax=axes[0], radius=0.075,
           highlight={"TF1", "TF2", "TF3"},
           title="gene regulatory network as a Bayesian network\nTFs (orange) → target genes → downstream genes")
draw_graph([("RAF", "MEK"), ("MEK", "ERK"), ("ERK", "RAF")], LOOP_POS, directed=True, ax=axes[1], radius=0.1,
           title="feedback loop RAF → MEK → ERK → RAF\n(a directed cycle: not allowed in a Bayesian network)")
draw_graph(G6_EDGES, G6_POS, directed=False, ax=axes[2],
           title="a six-variable undirected graph\n(a Markov random field: edges have no direction)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The left panel is a regulatory network written as a Bayesian network: to learn it we would
have to decide, for every linked pair, who regulates whom — from observational expression data alone, often impossible.
In the middle, three kinases form a loop. There is no way to order them so that every arrow points "forward" (no
topological order, Module 15), so the loop cannot be a Bayesian network. On the right is the six-variable undirected
graph that we use as the running example of Sections 2 and 6: an edge means "direct interaction", nothing more. In the
rest of the module the variables will be continuous — protein activities or gene expression levels — and the graph will
be encoded by a matrix.

---
## 2 · Cliques and graph separation

> 📍 **Where we are.** The MRF formula of Section 1 multiplies one potential per clique, and an undirected graph also
> tells us which variables are conditionally independent. Here we make both ideas concrete on the six-variable graph.
> In Module 16 you needed d-separation (with its tricky colliders) for directed graphs; for undirected graphs the rule
> is much simpler.

A **clique** is a subgraph in which *all nodes are completely connected* — every pair is joined by an edge. A
**maximal clique** cannot be enlarged by adding another node without losing that property. In our graph, single edges
such as $\{2,3\}$ and $\{4,6\}$ are cliques; the triangle $\{1,2,5\}$ is also a clique; the square $\{2,3,4,5\}$ is
**not** a clique because 2–4 and 3–5 are not connected. The next cell draws these three cases and then, because with
$K$ nodes there are only $2^K$ subsets, simply tests them all.

In [ ]:
# ── 2.0 Three node sets: cliques or not? ──
from matplotlib.patches import Ellipse


def ring_around(ax, nodes, pos, color, pad=0.11):
    """Draw a translucent ellipse around a set of nodes (to mark a candidate clique)."""
    xy = np.array([pos[n] for n in nodes])
    centre = xy.mean(axis=0)
    if len(nodes) == 2:                                   # a thin ellipse along the edge
        (xa, ya), (xb, yb) = xy
        ax.add_patch(Ellipse(centre, np.hypot(xb - xa, yb - ya) + 2 * pad, 1.6 * pad,
                             angle=np.degrees(np.arctan2(yb - ya, xb - xa)), fc=color, ec=color, alpha=0.2, lw=2, zorder=0))
    else:                                                 # a circle around the centroid
        r = np.max(np.hypot(*(xy - centre).T)) + pad
        ax.add_patch(Ellipse(centre, 2 * r, 2 * r, fc=color, ec=color, alpha=0.2, lw=2, zorder=0))


fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
for ax, sets, color, title in [(axes[0], [[2, 3], [4, 6], [1, 5]], "tab:green", "{2,3}, {4,6}, {1,5}: cliques (edges)"),
                               (axes[1], [[1, 2, 5]], "tab:green", "{1,2,5}: also a clique (a triangle)"),
                               (axes[2], [[2, 3, 4, 5]], "tab:red", "{2,3,4,5}: NOT a clique\n(2–4 and 3–5 are missing)")]:
    draw_graph(G6_EDGES, G6_POS, directed=False, ax=ax, title=title)
    for s in sets:
        ring_around(ax, s, G6_POS, color)
plt.tight_layout(); plt.show()

In [ ]:
# ── 2.1 Find every clique of the six-variable graph by brute force ──
def is_clique(nodes, edges) -> bool:
    """True if every pair of `nodes` is joined by an edge (a single node is a clique)."""
    edge_set = {frozenset(e) for e in edges}
    return all(frozenset(pair) in edge_set for pair in itertools.combinations(nodes, 2))


def all_cliques(nodes, edges) -> list:
    """All cliques, found by testing every subset of nodes (fine for small graphs: 2^K subsets)."""
    return [set(s) for size in range(1, len(nodes) + 1)
            for s in itertools.combinations(nodes, size) if is_clique(s, edges)]


def maximal_cliques(nodes, edges) -> list:
    """Cliques that are not contained in a larger clique."""
    cliques = all_cliques(nodes, edges)
    return [c for c in cliques if not any(c < d for d in cliques)]


cliques = all_cliques(G6_NODES, G6_EDGES)
for size in range(1, 5):
    print(f"cliques of size {size}:", [sorted(c) for c in cliques if len(c) == size])
print("maximal cliques:", [sorted(c) for c in maximal_cliques(G6_NODES, G6_EDGES)])
print("\n{2,3} is a clique?     ", is_clique([2, 3], G6_EDGES))
print("{1,2,5} is a clique?   ", is_clique([1, 2, 5], G6_EDGES))
print("{2,3,4,5} is a clique? ", is_clique([2, 3, 4, 5], G6_EDGES))

🔍 **What just happened?** The graph has 6 single-node cliques, 7 edge cliques (every edge is a clique) and one
triangle, $\{1,2,5\}$; there is no clique of size 4. The **maximal** cliques are $\{1,2,5\}$, $\{2,3\}$, $\{3,4\}$,
$\{4,5\}$ and $\{4,6\}$, so an MRF on this graph has the form

$$p(x_1,\dots,x_6)=\frac1Z\,\psi_{125}(x_1,x_2,x_5)\,\psi_{23}(x_2,x_3)\,\psi_{34}(x_3,x_4)\,\psi_{45}(x_4,x_5)\,\psi_{46}(x_4,x_6).$$

*In words:* the joint distribution is built only from "local" pieces, one per maximal clique; variables that never
share a clique interact only indirectly.

### 2.2 Reading conditional independence from an undirected graph

Every MRF satisfies three **Markov properties**:

- **Pairwise:** two *non-adjacent* variables are independent given all other variables, e.g. $X_6\perp X_3\mid X_1,X_2,X_4,X_5$.
- **Local:** a variable is independent of all others given its **neighbours**, e.g. $X_6\perp(X_1,X_2,X_3,X_5)\mid X_4$.
- **Global:** two sets $A$ and $B$ are independent given a set $C$ that **separates** them, e.g.
  $(X_6,X_4)\perp(X_1,X_2)\mid X_3,X_5$.

"$C$ separates $A$ from $B$" means: every path from a node of $A$ to a node of $B$ passes through $C$ — delete the nodes
of $C$ and $A$ can no longer reach $B$. That is an ordinary reachability question, which we answer by walking along
edges (a breadth-first search) without ever stepping on $C$.

In [ ]:
# ── 2.2 Graph separation: the three Markov properties on the six-variable graph ──
def neighbours(edges, node) -> set:
    """Nodes joined to `node` by an edge."""
    return {v for u, v in edges if u == node} | {u for u, v in edges if v == node}


def separated(edges, A, B, C) -> bool:
    """True if every path from a node in A to a node in B passes through the set C (graph separation).

    Start from A, walk along edges but never step onto a node of C, and report whether B was reached.
    """
    A, B, C = set(A), set(B), set(C)
    frontier, seen = list(A - C), set(A - C)
    while frontier:
        u = frontier.pop()
        if u in B:
            return False
        for v in neighbours(edges, u) - C - seen:
            seen.add(v)
            frontier.append(v)
    return True


queries = [("pairwise", {6}, {3}, {1, 2, 4, 5}),
           ("local", {6}, {1, 2, 3, 5}, {4}),
           ("global", {4, 6}, {1, 2}, {3, 5}),
           ("not a separator", {4, 6}, {1, 2}, {3}),
           ("nothing observed", {6}, {3}, set())]
for name, A, B, C in queries:
    print(f"{name:20s} {sorted(A)} ⊥ {sorted(B)} | {sorted(C)} ?  {separated(G6_EDGES, A, B, C)}")

🔍 **What just happened?** All three example statements are confirmed by separation. Conditioning on $X_3$ alone does
**not** separate $\{4,6\}$ from $\{1,2\}$ — the path $4-5-1$ remains open — and with nothing observed, $X_6$ and $X_3$
are connected through $X_4$. Notice how much simpler this is than d-separation in Module 16: in an undirected graph,
observing a node can only *block* paths, never open them (there are no colliders).

✏️ **Your turn.** Add the edge `(3, 6)` to `G6_EDGES` and re-run 2.1 and 2.2. Which maximal cliques appear, and which of
the five independence statements change? (Re-run cell 1.1 too if you want to see the new graph.)

---
## 3 · Discrete versus Gaussian Markov random fields

> 📍 **Where we are.** We know *what* an MRF is. Now: *which kind* this lecture topic focuses on, and why.
> Undirected models of **discrete** variables have no restriction on loops (unlike Bayesian networks, which forbid
> directed cycles), but inference and learning are expensive. For **continuous** variables, a Gaussian distribution can
> be used — a *Gaussian Markov random field*, called a **Gaussian graphical model** — and it is tractable.

For discrete variables, the potentials are tables and $Z$ is a sum over every configuration. With $K$ binary variables
that is $2^K$ terms; with the 11 Sachs proteins at three levels (low/medium/high, Module 17) it is $3^{11}=177{,}147$;
with 30 genes it is about $10^9$. Computing $Z$ is needed for likelihoods (learning) and for most queries (inference),
and for general graphs with loops no shortcut is known that always works. For a Gaussian (with mean 0 for simplicity),
the "potentials" are $\exp(-\tfrac12\Theta_{ii}x_i^2)$ for each node and $\exp(-\Theta_{ij}x_ix_j)$ for each edge, and
the integral has a closed form:

$$Z=\int\exp\!\Big(-\tfrac12\,\mathbf x^\top\Theta\,\mathbf x\Big)\,d\mathbf x=(2\pi)^{K/2}\,\det(\Theta)^{-1/2}.$$

*In words:* for a Gaussian MRF the normalising constant is one determinant of the $K\times K$ matrix $\Theta$ (defined
in Section 4) — a computation whose cost grows like $K^3$, not like $2^K$.

In [ ]:
# ── 3.1 Brute-force Z of a binary MRF: the cost doubles with every variable ──
def binary_chain_Z(K: int, coupling: float = 0.5) -> float:
    """Z of a chain-shaped binary MRF with ψ(x_i, x_{i+1}) = exp(coupling·x_i·x_{i+1}), x = ±1, by enumeration."""
    states = np.array(list(itertools.product([-1, 1], repeat=K)), dtype=np.int8)   # all 2^K configurations
    agreement = (states[:, :-1] * states[:, 1:]).sum(axis=1)                     # Σ_i x_i x_{i+1}
    return float(np.exp(coupling * agreement).sum())


print(" K    configurations   seconds")
for K in (8, 12, 16, 18):
    t0 = time.perf_counter()
    Z = binary_chain_Z(K)
    t_K = time.perf_counter() - t0
    print(f"{K:2d}  {2**K:>16,d}   {t_K:7.3f}")
for K in (30, 50, 100):
    print(f"{K:3d} {2.0**K:>16.2e}   ≈ {t_K * 2.0 ** (K - 18):.1e} (extrapolated)")
# A chain is a special (tree-shaped) graph with a closed form — check our brute force against it
print("\nchain K = 8: brute force", round(binary_chain_Z(8), 4), " closed form 2·(2cosh 0.5)^7 =",
      round(2 * (2 * np.cosh(0.5)) ** 7, 4))

In [ ]:
# ── 3.2 The Gaussian normalising constant: one log-determinant, even for 1,000 variables ──
K = 1000
Theta_chain = np.eye(K) - 0.4 * (np.eye(K, k=1) + np.eye(K, k=-1))       # a Gaussian chain of 1,000 "genes"
t0 = time.perf_counter()
sign, logdet = np.linalg.slogdet(Theta_chain)                             # stable log det(Θ)
logZ = 0.5 * K * np.log(2 * np.pi) - 0.5 * logdet
print(f"K = {K}: log Z = {logZ:.2f}, computed in {time.perf_counter() - t0:.3f} s")

🔍 **What just happened?** Every two extra binary variables quadruple the enumeration time; extrapolating the measured
time per configuration, 50 variables would take decades on this machine and 100 variables vastly longer than the age
of the universe (about $4\times10^{17}$ s). Chains and trees are special — they have closed forms and dynamic
programming (the trick behind the HMM algorithms of Module 19) — but general loopy graphs do not. The Gaussian
normalising constant for a 1,000-variable network, in contrast, takes a fraction of a second. **This is the reason the
rest of this module focuses on Gaussian graphical models.**

### 📘 Lecture notes — undirected graphical models (Markov random fields)

**Why undirected?** A Bayesian network of a regulatory network (TFs → targets → downstream genes) needs edge
directions, which observational data often cannot identify, and cannot contain feedback loops.

**Bayesian network vs Markov random field.** BN: directed edges, product of local conditional distributions. MRF:
undirected edges, normalised product of clique potentials,
$p(\mathbf x)=\frac1Z\prod_C\psi_C(\mathbf x_C)$, $Z=\sum_{\mathbf x}\prod_C\psi_C(\mathbf x_C)$.
*In words:* each clique gets a non-negative compatibility score; $Z$ makes the total probability 1.

**Cliques** (all nodes pairwise connected; maximal = cannot be enlarged). Six-node example (edges 1–2, 1–5, 2–3,
2–5, 3–4, 4–5, 4–6): $\{2,3\}$ and $\{1,2,5\}$ are cliques, $\{2,3,4,5\}$ is not.

**Markov properties** (read off by graph separation): *pairwise* — non-adjacent variables are independent given all
others, $X_6\perp X_3\mid X_1,X_2,X_4,X_5$; *local* — independent of all others given the neighbours,
$X_6\perp(X_1,X_2,X_3,X_5)\mid X_4$; *global* — sets separated by $C$ are independent given $C$,
$(X_6,X_4)\perp(X_1,X_2)\mid X_3,X_5$.

**Discrete vs Gaussian.** Discrete MRFs allow loops, but $Z$ sums over exponentially many configurations, so
inference and learning are expensive; for Gaussians $Z=(2\pi)^{K/2}\det(\Theta)^{-1/2}$ — the Gaussian graphical model.

> 🎓 **Exam tip.** Be able to list the maximal cliques of a small graph and to write the MRF factorisation over them. Remember
the difference from BNs: the MRF Markov blanket is just the set of neighbours, and observing a node can only block
paths (no colliders).

---
## 4 · Review: the multivariate Gaussian

> 📍 **Where we are.** A Gaussian graphical model *is* a multivariate Gaussian distribution, read in a particular way.
> Module 02 introduced the one-dimensional Gaussian and Module 03 vectors and matrices; here we put them together.
> The plan of the rest of the module follows the plan of the lecture topic: the **model** (review of the multivariate
> Gaussian; the GGM and its relationship to the multivariate Gaussian), **inference**, and **learning a model from
> data** (maximum likelihood, then sparsity).

For one variable, the normal distribution with mean $\mu$ and variance $\sigma^2$ is

$$p(x)=\mathcal N(\mu,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big).$$

*In words:* values near $\mu$ are most likely, and the probability falls off symmetrically; $\sigma$ sets the width.

🧮 **Math refresher — vectors, covariance matrices, inverse and determinant.** Collect $K$ variables (say the
phosphorylation levels of $K$ proteins in one cell) into a column vector $\mathbf x=(x_1,\dots,x_K)^\top$; the $^\top$
(**transpose**) turns a row into a column and vice versa. The **mean vector** $\boldsymbol\mu$ holds the $K$ means.
The **covariance matrix** $\Sigma$ is $K\times K$: its diagonal holds the variances $\sigma_i^2$, and the entry in row
$i$, column $j$ is the covariance $\sigma_{ij}$ of $x_i$ and $x_j$ (so $\sigma_{18}$ is the covariance of $x_1$ and
$x_8$, and the diagonal entries are often written $\sigma_1^2,\dots,\sigma_8^2$). $\Sigma$ is **symmetric** ($\sigma_{ij}=\sigma_{ji}$). The **inverse** $\Sigma^{-1}$ is the matrix that
"undoes" $\Sigma$: $\Sigma\Sigma^{-1}=I$, where the **identity matrix** $I$ has 1s on the diagonal and 0s elsewhere. The
**determinant** $\det(\Sigma)$ is a single number measuring how much $\Sigma$ stretches volumes; it is 0 exactly when
$\Sigma$ has no inverse. (All of these were introduced in Module 03.)

The multivariate normal density of $K$ variables, $p(x_1,\dots,x_K)=\mathcal N(\boldsymbol\mu,\Sigma)$, is

$$p(\mathbf x)=\mathcal N(\boldsymbol\mu,\Sigma)=\frac{1}{(2\pi)^{K/2}\det(\Sigma)^{1/2}}
\exp\!\Big(-\tfrac12(\mathbf x-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x-\boldsymbol\mu)\Big).$$

*In words:* the same bell shape, now in $K$ dimensions. The number $(\mathbf x-\boldsymbol\mu)^\top\Sigma^{-1}
(\mathbf x-\boldsymbol\mu)$ is a "distance from the mean" that accounts for correlations: moving *along* a correlation
is cheap, moving *against* it is expensive.

In [ ]:
# ── 4.1 The multivariate normal log-density, written straight from the formula ──
def mvn_logpdf(x, mu, Sigma):
    """log N(x; mu, Sigma) for every row of x (shape N × K), following the density formula above."""
    x = np.atleast_2d(x)
    K = len(mu)
    diff = x - mu                                          # deviations from the mean, one row per sample
    _, logdet = np.linalg.slogdet(Sigma)                   # log det(Σ), computed stably
    maha = np.sum((diff @ np.linalg.inv(Sigma)) * diff, axis=1)   # (x − μ)ᵀ Σ⁻¹ (x − μ) for every row
    return -0.5 * (K * np.log(2 * np.pi) + logdet + maha)


mu_demo = np.array([3.9, 3.3])                             # e.g. mean log-intensities of phospho-RAF and phospho-MEK
Sigma_demo = np.array([[0.50, 0.35], [0.35, 0.45]])
pts = np.array([[3.9, 3.3], [4.5, 3.8], [4.5, 2.8]])      # at the mean; both up; RAF up but MEK down
print("ours :", np.round(mvn_logpdf(pts, mu_demo, Sigma_demo), 6))
print("SciPy:", np.round(stats.multivariate_normal(mu_demo, Sigma_demo).logpdf(pts), 6))

🔍 **What just happened?** Our short function agrees with SciPy to six decimals. The second and third points are
both 0.6 units above the mean in RAF, but the second one (MEK also up) is much more likely than the third (MEK down):
with a positive covariance, "both up" goes *along* the correlation.

To **sample** from $\mathcal N(\boldsymbol\mu,\Sigma)$ we use a matrix square root. The **Cholesky factorisation**
writes $\Sigma=LL^\top$ with $L$ lower-triangular. If $\mathbf z$ is a vector of independent standard normals, then
$\mathbf x=\boldsymbol\mu+L\mathbf z$ has covariance $L\,I\,L^\top=\Sigma$:

$$\mathbf x=\boldsymbol\mu+L\mathbf z,\qquad \mathbf z\sim\mathcal N(\mathbf 0,I)\;\Rightarrow\;\mathbf x\sim\mathcal N(\boldsymbol\mu,LL^\top)=\mathcal N(\boldsymbol\mu,\Sigma).$$

*In words:* start from round, independent noise and stretch and shear it with $L$ until it has exactly the correlations
we want.

In [ ]:
# ── 4.2 Sampling with the Cholesky factor, and what the covariance does to the shape ──
def sample_mvn(mu, Sigma, n: int, rng) -> np.ndarray:
    """n draws (rows) from N(mu, Sigma): transform independent standard normals by the Cholesky factor L."""
    L = np.linalg.cholesky(Sigma)                          # lower-triangular, L @ L.T == Sigma
    z = rng.standard_normal((n, len(mu)))
    return mu + z @ L.T


covs = {"independent": [[1, 0], [0, 1]], "positively correlated": [[1, 0.8], [0.8, 1]],
        "negatively correlated": [[1, -0.6], [-0.6, 1]]}
g = np.linspace(-3.5, 3.5, 120)
gx, gy = np.meshgrid(g, g)
grid_pts = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.7))
for ax, (name, C) in zip(axes, covs.items()):
    C = np.array(C, float)
    dens = np.exp(mvn_logpdf(grid_pts, np.zeros(2), C)).reshape(gx.shape)
    xs = sample_mvn(np.zeros(2), C, 500, rng)
    ax.contour(gx, gy, dens, levels=6, colors="tab:blue", linewidths=1)
    ax.scatter(xs[:, 0], xs[:, 1], s=4, color="tab:orange", alpha=0.5)
    ax.set_title(f"{name}: Σ₁₂ = {C[0, 1]:+.1f}\nSIMULATED: correlation of 500 draws = {np.corrcoef(xs.T)[0, 1]:+.2f}")
    ax.set_xlabel("$x_1$ (standardised)"); ax.set_ylabel("$x_2$ (standardised)"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The contour lines of the density (blue) are circles when the variables are independent and
tilted ellipses when they are correlated; the 500 samples (orange) fill the ellipses, and their sample correlations are
close to the $\Sigma_{12}$ we asked for. A covariance matrix describes the *shape of the cloud* of cells.

✏️ **Your turn.** Change the positive correlation from 0.8 to 0.99 and to 1.0. What happens to the ellipse — and why does
`np.linalg.cholesky` refuse the value 1.0? (Hint: what is $\det\Sigma$ then?)

### The precision matrix

The inverse of the covariance matrix has a name of its own — the **precision matrix**:

$$\Theta=\Sigma^{-1},\qquad \Sigma\,\Theta=I.$$

*In words:* $\Theta$ is the matrix that undoes $\Sigma$. In the density it appears directly: the exponent is
$-\tfrac12(\mathbf x-\boldsymbol\mu)^\top\Theta(\mathbf x-\boldsymbol\mu)$, so $\Theta$, not $\Sigma$, says which
products $x_ix_j$ appear in the "energy" of a configuration. That is the key to the next section.

---
## 5 · The precision matrix is the graph

> 📍 **Where we are.** We can now write down, evaluate and sample a multivariate Gaussian. A **Gaussian graphical
> model** is a multivariate Gaussian whose graph is read off the precision matrix:

$$p(x_1,\dots,x_K)=\mathcal N(\boldsymbol\mu,\Theta^{-1}),\qquad \Theta_{ij}\neq0\iff\text{there is an edge between }x_i\text{ and }x_j.$$

*In words:* write the exponent $-\tfrac12\sum_i\sum_j\Theta_{ij}(x_i-\mu_i)(x_j-\mu_j)$ as a product of factors
$\exp(-\Theta_{ij}(x_i-\mu_i)(x_j-\mu_j))$, one per pair: each non-zero $\Theta_{ij}$ is an **edge potential** linking
$x_i$ and $x_j$, and $\Theta_{ij}=0$ means the pair has no factor of its own. So the GGM is an MRF whose potentials
live on edges, and its graph is the non-zero pattern of $\Theta$ — a Markov random field that satisfies all the
Markov properties of Section 2.

A classic teaching example is an 8-node ring $x_1-x_2-\dots-x_8-x_1$ with $p(x_1,\dots,x_8)=\mathcal N(\boldsymbol\mu,\Theta^{-1})$:
first its **adjacency matrix** (1 = edge, written with 1s on the diagonal), then an **edge-weighted adjacency
matrix** whose non-zero entries are the edge weights — and this weighted matrix is called $\Theta$. The next cell types
in those weights.

In [ ]:
# ── 5.1 The 8-node ring: adjacency matrix and edge-weighted Θ ──
NAMES8 = [f"x{k}" for k in range(1, 9)]
CYCLE8 = [("x1", "x2"), ("x2", "x3"), ("x3", "x4"), ("x4", "x5"), ("x5", "x6"), ("x6", "x7"), ("x7", "x8"), ("x8", "x1")]
THETA8_RAW = np.array([[1.0, 0.9, 0, 0, 0, 0, 0, 0.3],            # the edge-weighted adjacency matrix of the example
                       [0.9, 2.0, 1.0, 0, 0, 0, 0, 0],
                       [0, 1.0, 1.5, 0.8, 0, 0, 0, 0],
                       [0, 0, 0.8, 1.1, 0.9, 0, 0, 0],
                       [0, 0, 0, 0.9, 1.3, 0.7, 0, 0],
                       [0, 0, 0, 0, 0.7, 1.1, 0.9, 0],
                       [0, 0, 0, 0, 0, 0.9, 0.9, 1.0],
                       [0.3, 0, 0, 0, 0, 0, 1.0, 1.0]])
idx8 = {n: k for k, n in enumerate(NAMES8)}
ADJ8 = np.eye(8, dtype=int)                                       # adjacency matrix, written with 1s on the diagonal
for u, v in CYCLE8:
    ADJ8[idx8[u], idx8[v]] = ADJ8[idx8[v], idx8[u]] = 1
print("adjacency matrix:\n", ADJ8)
print("non-zero pattern of Θ equals the adjacency matrix?", np.array_equal(ADJ8, (THETA8_RAW != 0).astype(int)))
print("Θ symmetric?", np.allclose(THETA8_RAW, THETA8_RAW.T))
print("eigenvalues of the example Θ:", np.round(np.linalg.eigvalsh(THETA8_RAW), 3))

angles = np.pi / 2 + np.pi / 8 - np.arange(8) * np.pi / 4          # x1 top-left, then clockwise x8, x7, …, x2
POS8 = {n: (0.5 + 0.47 * np.cos(a), 0.5 + 0.47 * np.sin(a))
        for n, a in zip(["x1", "x8", "x7", "x6", "x5", "x4", "x3", "x2"], angles)}
fig, ax = plt.subplots(figsize=(4, 4))
draw_graph(CYCLE8, POS8, directed=False, ax=ax, edge_labels=[THETA8_RAW[idx8[u], idx8[v]] for u, v in CYCLE8],
           title="8-node ring GGM: edge weight = Θᵢⱼ")
plt.show()

🔍 **What just happened?** The zero pattern of the example's $\Theta$ is exactly the ring: $x_1$ is linked to $x_2$ and
$x_8$ only, and so on. **But** two eigenvalues of this matrix are negative. ⚠️ A precision (or covariance) matrix
must be **positive definite**: $\mathbf v^\top\Theta\mathbf v>0$ for every non-zero vector $\mathbf v$, equivalently all
**eigenvalues** (the "stretch factors" of the matrix along its special directions, Module 03) must be positive.
Otherwise $\exp(-\tfrac12\mathbf x^\top\Theta\mathbf x)$ would *grow* without bound in some direction and could not be
normalised. The example's numbers illustrate the *pattern* of a GGM, but they are not a valid distribution — off-diagonal
weights as large as the diagonal (e.g. row 7: diagonal 0.9, off-diagonals 0.9 and 1.0) are too strong.

The fix keeps the graph: adding a constant $c$ to every diagonal entry shifts every eigenvalue up by $c$ and leaves all
zeros in place. We add $c=0.7$ and use this repaired $\Theta$ for the rest of the module.

In [ ]:
# ── 5.2 Repair the example (same graph, valid Θ) and compare Θ with Σ = Θ⁻¹ ──
THETA8 = THETA8_RAW + 0.7 * np.eye(8)                     # every eigenvalue +0.7; the zero pattern is untouched
SIGMA8 = np.linalg.inv(THETA8)
print("eigenvalues of the repaired Θ:", np.round(np.linalg.eigvalsh(THETA8), 3))
print("Σ Θ = I ?", np.allclose(SIGMA8 @ THETA8, np.eye(8)))
print("zero entries in Θ:", int(np.sum(np.isclose(THETA8, 0))), "  zero entries in Σ:", int(np.sum(np.isclose(SIGMA8, 0))))


def show_matrix(ax, M, labels, title, fmt="{:.2f}", vmax=None):
    """Heat map of a matrix on a diverging colour scale centred at 0, with every entry written in."""
    vmax = vmax or np.abs(M).max()
    ax.imshow(M, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    for r in range(M.shape[0]):
        for c in range(M.shape[1]):
            ax.text(c, r, fmt.format(M[r, c]), ha="center", va="center", fontsize=6.5)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7, rotation=90)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.3))
show_matrix(axes[0], THETA8, NAMES8, "precision matrix Θ (8-node ring, repaired)\nzeros = missing edges", fmt="{:.1f}")
show_matrix(axes[1], SIGMA8, NAMES8, "covariance matrix Σ = Θ⁻¹\nno zeros at all")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The repaired $\Theta$ is positive definite and still has 40 zeros (the 20 missing pairs,
counted twice); its inverse $\Sigma$ has **none**. Every variable is correlated with every other, even with the variable
on the opposite side of the cycle, because influence travels around the ring. If we drew a network by connecting
correlated pairs, we would draw all 28 possible edges instead of the 8 real ones. *Correlation spreads along paths; the
precision matrix keeps only the direct links.*

### 📘 Lecture notes — the multivariate Gaussian and the Gaussian graphical model

**Multivariate normal** with mean vector $\boldsymbol\mu$ and covariance $\Sigma$ (variances on the diagonal):
$p(\mathbf x)=(2\pi)^{-K/2}\det(\Sigma)^{-1/2}\exp\!\big(-\tfrac12(\mathbf x-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x-\boldsymbol\mu)\big)$.
**Precision matrix** $\Theta=\Sigma^{-1}$, $\Sigma\Theta=I$.

**The GGM:** $p(x_1,\dots,x_K)=\mathcal N(\boldsymbol\mu,\Theta^{-1})$ with

$$\Theta_{ij}\neq0\iff\text{edge between }x_i\text{ and }x_j .$$

*In words:* the exponent is a sum of pairwise terms $\Theta_{ij}(x_i-\mu_i)(x_j-\mu_j)$, one factor per non-zero
entry — so the graph is the non-zero pattern of $\Theta$, and the GGM is a Markov random field.

**Example (8-node ring).** Adjacency matrix = the 8 ring edges (1s on the diagonal); the edge-weighted adjacency
matrix is $\Theta$. The typed-in weights show the *pattern* but are not positive definite; adding 0.7 to the
diagonal fixes that without changing the graph. $\Sigma=\Theta^{-1}$ then has **no zero at all**: correlation spreads
along paths, so reading edges off $\Sigma$ would draw all 28 pairs.

> 🎓 **Exam tip.** A valid precision matrix must be symmetric **and** positive definite; when you invent an example on an exam,
make the diagonal larger than the sum of the absolute off-diagonal entries in each row (diagonal dominance guarantees
positive definiteness).

### 5.3 The 5 × 5 example: precision versus covariance

A smaller example has two separate groups of variables, $\{y_1,y_2,y_3\}$ and $\{y_4,y_5\}$, with precision matrix

$$\Theta=\begin{bmatrix}1.3&0.3&0.8&0&0\\0.3&1.0&0&0&0\\0.8&0&1.2&0&0\\0&0&0&1.5&1.1\\0&0&0&1.1&0.9\end{bmatrix}.$$

*In words:* the matrix is **block-diagonal** (two blocks of non-zeros on the diagonal, zeros elsewhere), and its graph has the edges
$y_1-y_2$, $y_1-y_3$ and $y_4-y_5$. By the **global Markov property** — any two subsets of variables are conditionally
independent given a separating subset — $\{y_1,y_2,y_3\}\perp\{y_4,y_5\}$ given the empty set: no path joins them. The
inverse of a block-diagonal matrix is also block-diagonal, so **connected components can be found in both $\Theta$ and
$\Sigma$**. The difference lies *inside* a block.

In [ ]:
# ── 5.3 The 5 × 5 example: Θ shows the edges, Σ only the connected components ──
NAMES5 = ["y1", "y2", "y3", "y4", "y5"]
THETA5 = np.array([[1.3, 0.3, 0.8, 0.0, 0.0],
                   [0.3, 1.0, 0.0, 0.0, 0.0],
                   [0.8, 0.0, 1.2, 0.0, 0.0],
                   [0.0, 0.0, 0.0, 1.5, 1.1],
                   [0.0, 0.0, 0.0, 1.1, 0.9]])
SIGMA5 = np.linalg.inv(THETA5)
print("eigenvalues of Θ:", np.round(np.linalg.eigvalsh(THETA5), 3), "→ all positive")
print("Σ = Θ⁻¹ rounded to one decimal:\n", np.round(SIGMA5, 1) + 0.0)


def nonzero_edges(M, names, tol=1e-10) -> list:
    """Pairs (i < j) whose off-diagonal entry is non-zero."""
    return [(names[a], names[b]) for a, b in zip(*np.triu_indices(len(names), 1)) if abs(M[a, b]) > tol]


def connected_components(names, edges) -> list:
    """Groups of nodes linked by paths (breadth-first search from every not-yet-visited node)."""
    left, comps = list(names), []
    while left:
        comp, frontier = {left[0]}, [left[0]]
        while frontier:
            u = frontier.pop()
            for v in neighbours(edges, u) - comp:
                comp.add(v)
                frontier.append(v)
        comps.append(sorted(comp))
        left = [n for n in left if n not in comp]
    return comps


POS5 = {"y1": (0.2, 0.75), "y2": (0.05, 0.25), "y3": (0.42, 0.25), "y4": (0.65, 0.5), "y5": (0.95, 0.5)}
fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
for ax, (label, M) in zip(axes, [("Θ", THETA5), ("Σ", SIGMA5)]):
    E = nonzero_edges(M, NAMES5)
    print(f"non-zero pairs in {label}: {E}\n    connected components: {connected_components(NAMES5, E)}")
    draw_graph(E, POS5, directed=False, ax=ax, radius=0.08,
               title=f"pairs with a non-zero entry in {label}\n({len(E)} pairs, {len(connected_components(NAMES5, E))} connected components)")
    ax.set_ylim(0.0, 1.0)
plt.tight_layout(); plt.show()

🔍 **What just happened?** $\Sigma$ has the blocks
$\begin{bmatrix}1.5&-0.4&-1.0\\-0.4&1.1&0.3\\-1.0&0.3&1.5\end{bmatrix}$ and
$\begin{bmatrix}6.4&-7.9\\-7.9&10.7\end{bmatrix}$ and zeros between the blocks. Both matrices give the same two
connected components (inverting a block-diagonal matrix inverts each block separately), as the two graphs show. But
inside the first block the "graph" of $\Sigma$ is a triangle: $\Theta_{23}=0$ while $\Sigma_{23}\approx0.3$, so $y_2$ and
$y_3$ are **correlated**, yet there is **no edge** between them. Their correlation is entirely due to their shared neighbour
$y_1$ — once $y_1$ is known, $y_2$ tells us nothing more about $y_3$.

The quantity that measures this "direct" association is the **partial correlation** of $x_i$ and $x_j$ given all other
variables:

$$\rho_{ij\,\mid\,\text{rest}}=-\frac{\Theta_{ij}}{\sqrt{\Theta_{ii}\,\Theta_{jj}}}.$$

*In words:* rescale the precision entry by the two diagonal entries and flip its sign; the result is the correlation that
remains between $x_i$ and $x_j$ after the influence of every other variable has been removed (we prove this in
Section 6). It is zero exactly when $\Theta_{ij}=0$.

In [ ]:
# ── 5.4 Correlation versus partial correlation in the 5 × 5 example ──
def corr_from_cov(Sigma):
    """Correlation matrix: Σᵢⱼ / √(Σᵢᵢ Σⱼⱼ)."""
    d = np.sqrt(np.diag(Sigma))
    return Sigma / np.outer(d, d)


def partial_corr(Theta):
    """Partial correlation of every pair given all other variables: −Θᵢⱼ / √(Θᵢᵢ Θⱼⱼ) (1 on the diagonal)."""
    d = np.sqrt(np.diag(Theta))
    P = -Theta / np.outer(d, d) + 0.0                      # + 0.0 turns −0.0 into 0.0 for printing
    np.fill_diagonal(P, 1.0)
    return P


R5, P5 = corr_from_cov(SIGMA5), partial_corr(THETA5)
for a, b in [(0, 1), (0, 2), (1, 2), (3, 4), (0, 3)]:
    print(f"{NAMES5[a]}–{NAMES5[b]}:  Θ = {THETA5[a, b]:+.2f}   correlation = {R5[a, b]:+.3f}"
          f"   partial correlation = {P5[a, b]:+.3f}")
fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
show_matrix(axes[0], THETA5, NAMES5, "precision Θ", fmt="{:.1f}")
show_matrix(axes[1], SIGMA5, NAMES5, "covariance Σ = Θ⁻¹", fmt="{:.1f}")
show_matrix(axes[2], R5, NAMES5, "correlation", vmax=1)
show_matrix(axes[3], P5, NAMES5, "partial correlation", vmax=1)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The correlation matrix (third panel) is non-zero for $y_2$–$y_3$ (+0.227); the partial
correlation matrix (fourth panel) is zero there, and has the same zero pattern as $\Theta$. Within the second block the
partial correlation of $y_4$ and $y_5$ is $-1.1/\sqrt{1.5\times0.9}=-0.947$: a strong *negative* direct association.

⚠️ **Sign pitfall.** A *positive* precision entry means a *negative* partial correlation, and vice versa
($\Theta_{12}=+0.3$ gives $\rho_{12\mid\text{rest}}=-0.263$). Both teaching examples (the ring and the 5 × 5 matrix)
have only positive off-diagonal $\Theta$ entries, so every edge in them is a negative partial correlation. Always
convert to partial correlations before interpreting signs.

✏️ **Your turn.** Set `THETA5[1, 2] = THETA5[2, 1] = 0.4` (an edge $y_2$–$y_3$), check that $\Theta$ is still positive
definite, and re-run 5.3–5.4. Does $\Sigma$ change only in the $y_2$–$y_3$ entry, or everywhere in the first block? Why?

---
## 6 · The Markov properties, verified numerically

> 📍 **Where we are.** Section 2 read independences off a graph; Section 5 claimed that zeros of $\Theta$ are missing
> edges. Now we check, with actual Gaussian distributions, that the graph of $\Theta$ obeys all three Markov properties.

The three properties of Section 2 reappear for the GGM:

- **Local:** *a variable is conditionally independent of all other variables given its neighbours.* Which variables
  must we condition on to make $x_1$ of the 8-node ring independent of the rest? Its two neighbours:
  $X_1\perp(X_3,\dots,X_7)\mid(X_2,X_8)$. The **Markov blanket** of a node in an MRF is simply its set of neighbours —
  different from a Bayesian network, where the blanket also contains children and co-parents (Module 16).
- **Pairwise:** *any two non-adjacent variables are conditionally independent given all other variables.* A classic
  example is the graph of the exam marks of students in five subjects (Mechanics, Vectors, Algebra, Analysis,
  Statistics; Mardia, Kent & Bibby 1979, analysed by Whittaker 1990), drawn in the next cell: Algebra sits in the
  middle of a "butterfly", and Statistics and Mechanics are independent given all the other marks.
- **Global:** *two sets are conditionally independent given a separating set* — the 5 × 5 example of Section 5.3, where
  the empty set separates $\{y_1,y_2,y_3\}$ from $\{y_4,y_5\}$.

In [ ]:
# ── 6.0 The exam-marks butterfly: Statistics and Mechanics are independent given the other marks ──
SUBJ = ["Mech", "Vect", "Alg", "Anal", "Stat"]          # Mechanics, Vectors, Algebra, Analysis, Statistics
MARK_EDGES = [("Mech", "Vect"), ("Mech", "Alg"), ("Vect", "Alg"), ("Alg", "Anal"), ("Alg", "Stat"), ("Anal", "Stat")]
ixs = {s: k for k, s in enumerate(SUBJ)}
THETA_MARKS = np.eye(5)
THETA_MARKS[ixs["Alg"], ixs["Alg"]] = 1.6                 # Algebra has 4 neighbours: a larger diagonal keeps Θ PD
for u, v in MARK_EDGES:
    THETA_MARKS[ixs[u], ixs[v]] = THETA_MARKS[ixs[v], ixs[u]] = -0.3   # Θ < 0: positive partial correlation
rng_marks = np.random.default_rng(SEED + 60)
marks = 50 + 12 * sample_mvn(np.zeros(5), np.linalg.inv(THETA_MARKS), 88, rng_marks)   # 88 SIMULATED students
R_marks = np.corrcoef(marks.T)
P_marks = partial_corr(np.linalg.inv(np.cov(marks.T)))
P_marks_true = partial_corr(THETA_MARKS)
print("pair        edge?  true partial corr   sample corr   sample partial corr  (88 students)")
for a, b in itertools.combinations(range(5), 2):
    edge = (SUBJ[a], SUBJ[b]) in MARK_EDGES or (SUBJ[b], SUBJ[a]) in MARK_EDGES
    print(f"{SUBJ[a]:>4s}–{SUBJ[b]:<4s}   {str(edge):5s}   {P_marks_true[a, b]:+.3f}            {R_marks[a, b]:+.3f}"
          f"         {P_marks[a, b]:+.3f}")

MARK_POS = {"Vect": (0.1, 0.85), "Mech": (0.1, 0.15), "Alg": (0.5, 0.5), "Anal": (0.9, 0.85), "Stat": (0.9, 0.15)}
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.6))
draw_graph(MARK_EDGES, MARK_POS, directed=False, ax=axes[0], highlight={"Mech", "Stat"}, radius=0.09,
           title="exam-marks graph: Mech ⊥ Stat | all other marks")
show_matrix(axes[1], P_marks, SUBJ, "SIMULATED marks of 88 students:\nsample partial correlations", vmax=1)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The simulated marks of 88 students are all positively correlated with one another — good
students tend to be good at everything — including the non-adjacent pairs such as Mechanics and Statistics. But their
*partial* correlations tell the butterfly apart: the six edges have clearly positive partial correlations, while the four
non-adjacent pairs (true value exactly 0) give sample values that are small and scattered around zero, as expected
from 88 students (one standard error is about $1/\sqrt{88}\approx0.11$). Given Algebra (and the rest), knowing a
student's Mechanics mark tells us nothing more about their Statistics mark: **Algebra separates the two wings**.

For Gaussians, conditional independence can be checked exactly with one formula. If we split the variables into a set
$A$ we care about and a set $C$ we condition on,

$$\operatorname{Cov}(\mathbf x_A\mid\mathbf x_C)=\Sigma_{AA}-\Sigma_{AC}\,\Sigma_{CC}^{-1}\,\Sigma_{CA}.$$

*In words:* start from the covariance of $A$ and subtract the part that is explained by $C$. Two remarkable Gaussian
facts: the result does not depend on the *values* $\mathbf x_C$ we observed, and for Gaussians *zero conditional
covariance is the same as conditional independence*. ($\Sigma_{AC}$ means "the rows of $A$ and columns of $C$ of
$\Sigma$"; the formula is derived in Section 7.)

In [ ]:
# ── 6.1 Conditional covariance: check the local and pairwise Markov properties of the 8-node GGM ──
rng = np.random.default_rng(SEED + 6)


def conditional_cov(Sigma, A, C):
    """Covariance of x_A given x_C for a Gaussian: Σ_AA − Σ_AC Σ_CC⁻¹ Σ_CA."""
    A, C = list(A), list(C)
    S_AA = Sigma[np.ix_(A, A)]
    if not C:
        return S_AA
    S_AC, S_CC = Sigma[np.ix_(A, C)], Sigma[np.ix_(C, C)]
    return S_AA - S_AC @ np.linalg.solve(S_CC, S_AC.T)


def cond_corr(Sigma, a, b, C):
    """Correlation of x_a and x_b given x_C (a partial correlation)."""
    c = conditional_cov(Sigma, [a, b], C)
    return c[0, 1] / np.sqrt(c[0, 0] * c[1, 1])


far = [idx8[n] for n in ["x3", "x4", "x5", "x6", "x7"]]
cc = conditional_cov(SIGMA8, [idx8["x1"]] + far, [idx8["x2"], idx8["x8"]])
print(f"local:    max |Cov(x1, x3…x7 | x2, x8)| = {np.abs(cc[0, 1:]).max():.1e}")
cc = conditional_cov(SIGMA8, [idx8["x1"]] + far, [idx8["x2"]])
print(f"          max |Cov(x1, x3…x7 | x2)|     = {np.abs(cc[0, 1:]).max():.3f}   (x8 left out → not independent)")

P8 = partial_corr(THETA8)
worst = 0.0
print("pairwise: partial correlation by conditioning versus −Θij/√(ΘiiΘjj)")
for a, b in itertools.combinations(range(8), 2):
    rest = [k for k in range(8) if k not in (a, b)]
    pc = cond_corr(SIGMA8, a, b, rest)
    worst = max(worst, abs(pc - P8[a, b]))
    if (a, b) in [(0, 1), (0, 2), (0, 4), (2, 3)]:
        print(f"   {NAMES8[a]}–{NAMES8[b]}: conditioning {pc:+.4f}   from Θ {P8[a, b]:+.4f}   edge? {THETA8[a, b] != 0}")
print(f"   largest disagreement over all 28 pairs: {worst:.1e}")
rest = [k for k in range(8) if k not in (0, 2)]
print("Cov(x1, x3 | rest) equals the inverse of Θ's 2×2 block?",
      np.allclose(conditional_cov(SIGMA8, [0, 2], rest), np.linalg.inv(THETA8[np.ix_([0, 2], [0, 2])])))

🔍 **What just happened?** Given its two neighbours $x_2$ and $x_8$, $x_1$ is uncorrelated with every other variable
(conditional covariances of order $10^{-16}$, i.e. zero up to rounding) — the local Markov property. Conditioning on
$x_2$ alone is not enough, because information still flows the other way round the ring through $x_8$. For all 28 pairs,
the partial correlation obtained by conditioning on the other six variables equals $-\Theta_{ij}/\sqrt{\Theta_{ii}
\Theta_{jj}}$, and it is zero exactly for the 20 non-adjacent pairs — the pairwise Markov property.

**Why the formula holds.** The last line shows the key fact: the covariance of a pair $(x_i,x_j)$ given *all other*
variables is the inverse of the $2\times2$ block of $\Theta$,

$$\operatorname{Cov}\!\big((x_i,x_j)\mid\text{rest}\big)=\begin{bmatrix}\Theta_{ii}&\Theta_{ij}\\\Theta_{ij}&\Theta_{jj}\end{bmatrix}^{-1}
=\frac{1}{\Theta_{ii}\Theta_{jj}-\Theta_{ij}^2}\begin{bmatrix}\Theta_{jj}&-\Theta_{ij}\\-\Theta_{ij}&\Theta_{ii}\end{bmatrix}.$$

*In words:* conditioning on everything else leaves a 2-variable Gaussian whose precision matrix is just the $\{i,j\}$
block of $\Theta$. Its off-diagonal covariance is proportional to $-\Theta_{ij}$, so it vanishes exactly when
$\Theta_{ij}=0$; dividing by the square roots of the two diagonal entries gives the partial-correlation formula.

### 6.2 The global property: zero conditional covariance ⇔ graph separation

We now take the six-node graph of Section 2, put random weights on its 7 edges (making sure $\Theta$ is positive definite),
and test **every** pair of variables against **every** possible conditioning set.

In [ ]:
# ── 6.2 Global Markov property: zero conditional covariance ⇔ graph separation (exhaustive check) ──
def random_precision(edges, K, rng, strength=(0.2, 0.5)):
    """A random positive-definite Θ whose non-zero off-diagonal entries are exactly `edges` (0-based pairs)."""
    Theta = np.zeros((K, K))
    for u, v in edges:
        Theta[u, v] = Theta[v, u] = rng.choice([-1, 1]) * rng.uniform(*strength)
    np.fill_diagonal(Theta, np.abs(Theta).sum(axis=1) + 0.5)      # diagonal dominance ⇒ positive definite
    return Theta


edges0 = [(u - 1, v - 1) for u, v in G6_EDGES]                     # 0-based indices for the matrix
Theta6 = random_precision(edges0, 6, rng)
Sigma6 = np.linalg.inv(Theta6)
agree = total = 0
for a, b in itertools.combinations(range(6), 2):
    rest = [k for k in range(6) if k not in (a, b)]
    for size in range(len(rest) + 1):
        for C in itertools.combinations(rest, size):
            zero_cov = abs(conditional_cov(Sigma6, [a, b], C)[0, 1]) < 1e-10
            sep = separated(G6_EDGES, {a + 1}, {b + 1}, {c + 1 for c in C})
            agree += zero_cov == sep
            total += 1
print(f"zero conditional covariance ⇔ graph separation in {agree} of {total} (pair, conditioning set) cases")
blk = conditional_cov(Sigma6, [5, 3, 0, 1], [2, 4])                # (X6, X4) versus (X1, X2) given X3, X5
print("global example: Cov((X6, X4), (X1, X2) | X3, X5) =\n", np.round(blk[:2, 2:], 12) + 0.0)
blk = conditional_cov(Sigma6, [5, 3, 0, 1], [2])                   # given X3 only
print("given X3 only:\n", np.round(blk[:2, 2:], 4))

🔍 **What just happened?** In all 240 cases, the conditional covariance is zero exactly when the conditioning set
separates the two variables in the graph. The global-property example $(X_6,X_4)\perp(X_1,X_2)\mid X_3,X_5$ gives a block of
exact zeros; conditioning on $X_3$ alone leaves the path through $X_5$ open and the block is non-zero.

⚠️ "Separated ⇒ independent" holds for *every* Gaussian with this graph. The converse ("not separated ⇒ dependent") holds
for *generic* weights; with carefully tuned weights two paths could cancel exactly (this assumption is called
*faithfulness*, as for Bayesian networks in Module 16).

### 6.3 The same fact in data

Finally, the most practical version: simulate many "cells" from the 8-node GGM, and remove the influence of the
neighbours by **regression** — keep only the part of $x_1$ (and of $x_3$) that the neighbours cannot predict, the
**residual** (Module 06). If $x_1\perp x_3\mid x_2,x_8$, the residuals should be uncorrelated.

In [ ]:
# ── 6.3 In data: x1 and x3 are correlated, but not once x2 and x8 are accounted for ──
X8 = sample_mvn(np.zeros(8), SIGMA8, 20_000, rng)          # 20,000 SIMULATED cells from the 8-node GGM


def residual(y, Z):
    """Residual of the least-squares regression of y on the columns of Z (with an intercept)."""
    Z1 = np.c_[np.ones(len(y)), Z]
    coef, *_ = np.linalg.lstsq(Z1, y, rcond=None)
    return y - Z1 @ coef


x1, x3 = X8[:, 0], X8[:, 2]
panels = [("raw values", x1, x3, []),
          ("given x2", residual(x1, X8[:, [1]]), residual(x3, X8[:, [1]]), [1]),
          ("given x2 and x8", residual(x1, X8[:, [1, 7]]), residual(x3, X8[:, [1, 7]]), [1, 7])]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, (title, u, v, C) in zip(axes, panels):
    r = np.corrcoef(u, v)[0, 1]
    theory = cond_corr(SIGMA8, 0, 2, C)
    ax.scatter(u[:2000], v[:2000], s=3, alpha=0.4, color="tab:blue")
    ax.set_title(f"SIMULATED: x1 vs x3, {title}\nsample r = {r:+.3f} (theory {theory:+.3f})")
    ax.set_xlabel("x1 (or its residual)"); ax.set_ylabel("x3 (or its residual)")
    print(f"{title:16s} sample correlation {r:+.3f}   theory {theory:+.3f}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the raw SIMULATED data $x_1$ and $x_3$ are clearly correlated (they share the neighbour
$x_2$). Removing what $x_2$ explains shrinks the correlation to almost nothing: in theory 0.025 remains, because the
long way round the ring ($x_3-x_4-\dots-x_8-x_1$) is still open, but a path of six edges transmits very little.
Removing both $x_2$ and $x_8$ makes the theoretical correlation exactly 0. With 20,000 cells a sample correlation
wobbles by about $\pm1/\sqrt{20{,}000}\approx\pm0.007$ (one standard error), so the two sample values (+0.008 and
−0.018), each about 0.017 below the theory, are a modest fluctuation of this particular sample (about 2.5 standard
errors, shared by both panels because they use the same cells). A partial correlation *is* a correlation of residuals.

✏️ **Your turn.** Repeat the experiment for $x_1$ and $x_5$ (the opposite side of the ring). Which minimal set of
variables do you have to condition on to make them independent? Check your answer with `separated` (using the names
`"x1"`, … and `CYCLE8`) and with `cond_corr`.

### 📘 Lecture notes — Markov properties of a GGM; precision versus covariance

**Local:** a variable is independent of all others given its neighbours — the Markov blanket of a node in an
undirected graph is all its neighbours (in a BN: parents, children, co-parents). Ring: $X_1\perp(X_3,\dots,X_7)\mid(X_2,X_8)$.
**Pairwise:** non-adjacent variables are independent given all others — exam marks: Statistics $\perp$ Mechanics
$\mid$ the other marks (Algebra separates the two wings). **Global:** subsets are independent given a separating
subset — in the 5 × 5 example both $\Theta$ and $\Sigma$ are block-diagonal (connected components are visible in
both), but only $\Theta$ shows that $y_2$ and $y_3$ are not linked ($\Theta_{23}=0$, $\Sigma_{23}\approx0.3$).

**Tools.** $\operatorname{Cov}(\mathbf x_A\mid\mathbf x_C)=\Sigma_{AA}-\Sigma_{AC}\Sigma_{CC}^{-1}\Sigma_{CA}$ (zero ⇔
conditional independence for Gaussians); the covariance of a pair given all the rest is the inverse of the
$2\times2$ block of $\Theta$, hence

$$\rho_{ij\mid\text{rest}}=-\frac{\Theta_{ij}}{\sqrt{\Theta_{ii}\Theta_{jj}}}.$$

*In words:* rescale the precision entry and flip its sign (5 × 5 example: $y_1,y_3$ give $-0.641$). A partial
correlation is the correlation of the residuals after regressing both variables on the others.

> 🎓 **Exam tip.** A frequent exam error is to read edges off the *covariance* matrix. Zeros of $\Sigma$ mean *marginal*
independence (e.g. different connected components); zeros of $\Theta$ mean *conditional* independence given all other
variables, i.e. missing edges. And mind the sign: positive $\Theta_{ij}$ = negative partial correlation.

---
## 7 · Inference: marginals, conditionals — and regression

> 📍 **Where we are.** Inference means answering questions such as "what is the distribution of $x_1$?" or "what do we
> expect for $x_1$ if we have measured all the others?". For Gaussians both answers are again Gaussian, and the second
> one turns out to be a linear regression — the model of Module 06.

**Marginals.** How can we infer $p(x_1)$, or $p(x_1,x_2)$, from $p(x_1,\dots,x_8)=\mathcal N(\boldsymbol\mu,\Sigma)$? To get
the distribution of a subset of variables, just *pick the corresponding entries*: $p(x_1)=\mathcal N(\mu_1,\sigma_1^2)$ and

$$p(x_1,x_2)=\mathcal N\!\left(\begin{bmatrix}\mu_1\\\mu_2\end{bmatrix},\begin{bmatrix}\sigma_1^2&\sigma_{12}\\\sigma_{12}&\sigma_2^2\end{bmatrix}\right).$$

*In words:* marginalising (integrating out) the other variables of a Gaussian simply deletes their rows and columns from
$\boldsymbol\mu$ and $\Sigma$ — no integral needs to be computed.

**Conditionals.** By the definition of conditional probability, $p(x_1\mid x_2)=p(x_1,x_2)/p(x_2)$ — and we know how to
compute both the numerator and the denominator: they are the marginals just described. Working out the ratio gives
another Gaussian:

$$p(x_1\mid x_2)=\mathcal N\!\Big(\mu_1+\frac{\sigma_{12}}{\sigma_2^2}(x_2-\mu_2),\;\sigma_1^2-\frac{\sigma_{12}^2}{\sigma_2^2}\Big).$$

*In words:* knowing $x_2$ shifts our expectation for $x_1$ by an amount proportional to how far $x_2$ is from its mean
(more if the covariance is large), and it always *reduces* the variance — by more when the two are strongly correlated.

In [ ]:
# ── 7.1 Marginal and conditional distributions of the 8-node GGM, checked numerically ──
rng = np.random.default_rng(SEED + 7)
mu8 = np.linspace(1.0, 4.5, 8)                            # any mean vector will do
big = sample_mvn(mu8, SIGMA8, 200_000, rng)
print("marginal of (x1, x2): just read off μ and Σ")
print("   formula mean", np.round(mu8[:2], 3), "  sample mean", np.round(big[:, :2].mean(axis=0), 3))
print("   formula cov ", np.round(SIGMA8[:2, :2].ravel(), 3), "  sample cov", np.round(np.cov(big[:, :2].T).ravel(), 3))

print("\nconditional p(x1 | x2) = p(x1, x2) / p(x2) versus the closed-form Gaussian")
s11, s12, s22 = SIGMA8[0, 0], SIGMA8[0, 1], SIGMA8[1, 1]
for x1v, x2v in [(1.0, 1.5), (2.0, 0.5), (0.0, 3.0)]:
    log_ratio = mvn_logpdf([[x1v, x2v]], mu8[:2], SIGMA8[:2, :2])[0] - stats.norm(mu8[1], np.sqrt(s22)).logpdf(x2v)
    m, v = mu8[0] + s12 / s22 * (x2v - mu8[1]), s11 - s12 ** 2 / s22
    print(f"   x1 = {x1v}, x2 = {x2v}:  log p(x1,x2) − log p(x2) = {log_ratio:.6f}"
          f"   log N(x1; {m:.3f}, {v:.3f}) = {stats.norm(m, np.sqrt(v)).logpdf(x1v):.6f}")

🔍 **What just happened?** The sample mean and covariance of $(x_1,x_2)$ over 200,000 simulated draws match the
entries read straight off $\boldsymbol\mu$ and $\Sigma$, and the ratio of the joint and marginal densities equals the
closed-form conditional Gaussian at every test point.

### 7.2 One variable given all the others

Applying the same reasoning to $x_1$ and the seven other variables of the 8-node model gives the conditional
distribution of a single variable given all the others:

$$p(x_1\mid x_2,\dots,x_8)=\mathcal N\!\Big(\mu_1+(\mathbf x_{2:8}-\boldsymbol\mu_{2:8})^\top\Sigma_{2:8}^{-1}\Sigma_{2:8,1},\;\;
\sigma_{1}^2-\Sigma_{1,2:8}\,\Sigma_{2:8}^{-1}\,\Sigma_{2:8,1}\Big).$$

*In words:* the expected value of $x_1$ is its own mean plus a weighted sum of how far each other variable is from its
mean; the weights are $\Sigma_{2:8}^{-1}\Sigma_{2:8,1}$ (here $\Sigma_{2:8}$ is the $7\times7$ covariance of
$x_2,\dots,x_8$ and $\Sigma_{2:8,1}$ their covariances with $x_1$), and the variance shrinks by the part the others explain.

Compare this with the linear-regression model of Module 06, $p(y\mid X)=\mathcal N(\beta_0+X\beta,\sigma^2)$: the
conditional **is** a regression of $x_1$ on the other variables — $\mu_1$ plays the role of the "prior mean" $\beta_0$,
the other observations $\mathbf x_{2:8}-\boldsymbol\mu_{2:8}$ are the predictors $X$, and the weights are the "effects of
the other observations" $\beta$. There is an even nicer way to write the weights, using $\Theta$.

**Derivation by "completing the square".** Take $\boldsymbol\mu=\mathbf 0$ for readability. The log-density is
$-\tfrac12\mathbf x^\top\Theta\mathbf x+\text{const}=-\tfrac12\sum_i\sum_j\Theta_{ij}x_ix_j+\text{const}$. Keep only
the terms that contain $x_1$ (everything else is a constant once $x_2,\dots,x_8$ are fixed):

$$\log p(x_1\mid\mathbf x_{-1})=-\tfrac12\Theta_{11}x_1^2-x_1\sum_{j\neq1}\Theta_{1j}x_j+\text{const}.$$

*In words:* as a function of $x_1$ alone, the log-density is a downward parabola — so $x_1$ given the rest is Gaussian.

A one-dimensional Gaussian $\mathcal N(m,v)$ has log-density $-\frac{x^2}{2v}+\frac{m}{v}x+\text{const}$. Matching the
$x_1^2$ terms gives $1/v=\Theta_{11}$, and matching the $x_1$ terms gives $m/v=-\sum_{j\ne1}\Theta_{1j}x_j$. Hence

$$x_1\mid\mathbf x_{-1}\;\sim\;\mathcal N\Big(-\sum_{j\neq1}\frac{\Theta_{1j}}{\Theta_{11}}\,x_j,\;\;\frac{1}{\Theta_{11}}\Big).$$

*In words:* the regression coefficient of $x_j$ is $-\Theta_{1j}/\Theta_{11}$, and the residual variance is
$1/\Theta_{11}$. **Non-neighbours get a coefficient of exactly zero** — to predict a variable you only need its
neighbours, which is the local Markov property again. (With a non-zero mean, replace each $x_j$ by $x_j-\mu_j$ and add
$\mu_1$.)

In [ ]:
# ── 7.2 The bridge identity: Σ-weights = −Θ1j/Θ11, conditional variance = 1/Θ11 ──
rest = list(range(1, 8))
beta_cov = np.linalg.solve(SIGMA8[np.ix_(rest, rest)], SIGMA8[rest, 0])     # Σ_{2:8}⁻¹ Σ_{2:8,1}
var_cov = SIGMA8[0, 0] - SIGMA8[0, rest] @ beta_cov                       # σ₁² − Σ_{1,2:8} Σ_{2:8}⁻¹ Σ_{2:8,1}
beta_prec = -THETA8[0, rest] / THETA8[0, 0]
print("weights from Σ          :", np.round(beta_cov, 4))
print("weights −Θ1j/Θ11         :", np.round(beta_prec, 4) + 0.0)
print(f"conditional variance from Σ: {var_cov:.4f}      1/Θ11 = {1 / THETA8[0, 0]:.4f}")

🔍 **What just happened?** The two formulas give identical weights: $x_1$ depends only on $x_2$ (weight
$-0.9/1.7=-0.529$) and $x_8$ ($-0.3/1.7=-0.176$); the five non-neighbours get exactly zero. The conditional variance is
$1/\Theta_{11}=1/1.7=0.588$. Now let us see whether ordinary least squares on *data* finds the same numbers.

In [ ]:
# ── 7.3 Regress every node on all the others (SIMULATED data) and compare with −Θjk/Θjj ──
Xs = sample_mvn(mu8, SIGMA8, 5000, rng)
B_ols, resid_var, intercepts = np.zeros((8, 8)), np.zeros(8), np.zeros(8)
for j in range(8):
    others = [k for k in range(8) if k != j]
    Z1 = np.c_[np.ones(len(Xs)), Xs[:, others]]           # design matrix with an intercept column (Module 06)
    coef, *_ = np.linalg.lstsq(Z1, Xs[:, j], rcond=None)
    intercepts[j], B_ols[j, others] = coef[0], coef[1:]
    resid_var[j] = np.mean((Xs[:, j] - Z1 @ coef) ** 2)
B_theory = -THETA8 / np.diag(THETA8)[:, None]             # row j holds −Θjk / Θjj
np.fill_diagonal(B_theory, 0.0)
print("node x1:  OLS coefficients on x2…x8:", np.round(B_ols[0, 1:], 3))
print("          theory −Θ1k/Θ11          :", np.round(B_theory[0, 1:], 3) + 0.0)
print("residual variances:", np.round(resid_var, 3))
print("1/Θjj             :", np.round(1 / np.diag(THETA8), 3))
print(f"intercept of x1: {intercepts[0]:.3f}    theory μ1 − Σk βk μk = {mu8[0] - B_theory[0] @ mu8:.3f}")

off = ~np.eye(8, dtype=bool)
edge_mask = (THETA8 != 0) & off
fig, ax = plt.subplots(figsize=(4.6, 4.1))
ax.scatter(B_theory[off & ~edge_mask], B_ols[off & ~edge_mask], s=18, color="tab:blue", label="non-neighbours (theory 0)")
ax.scatter(B_theory[edge_mask], B_ols[edge_mask], s=18, color="tab:red", label="neighbours (edges)")
lim = np.abs(B_theory).max() * 1.15
ax.plot([-lim, 0.2], [-lim, 0.2], "k--", lw=1)
ax.set_xlabel("theory  −Θⱼₖ / Θⱼⱼ"); ax.set_ylabel("OLS coefficient (N = 5,000)")
ax.set_title("56 node-wise regressions (SIMULATED):\nregression coefficients = scaled precision entries")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

🔍 **What just happened?** For every one of the 8 regressions (56 coefficients), ordinary least squares on 5,000
simulated samples recovers $-\Theta_{jk}/\Theta_{jj}$: the red points (neighbours) lie on the diagonal, the blue points
(non-neighbours) cluster at 0 — but, being estimates from finite data, they are *not exactly* 0. The residual variances
match $1/\Theta_{jj}$. The intercept (2.73) is close to $\mu_1-\sum_k\beta_k\mu_k=2.59$ but less precise, because it
absorbs the small errors of all seven coefficients multiplied by means as large as 4.5.

The dictionary between the two models:

| Regression $p(y\mid X)=\mathcal N(\beta_0+X\beta,\sigma^2)$ | GGM conditional $p(x_1\mid\mathbf x_{2:8})$ |
|---|---|
| response $y$ | the variable $x_1$ |
| $\beta_0$ — the "prior mean" | $\mu_1$ (exactly, when the predictors are centred, $\mathbf x_{2:8}-\boldsymbol\mu_{2:8}$; otherwise the intercept is $\mu_1-\boldsymbol\mu_{2:8}^\top\beta$) |
| predictors $X$ — "other observations" | $\mathbf x_{2:8}-\boldsymbol\mu_{2:8}$ |
| coefficients $\beta$ — "effects of other observations" | $\Sigma_{2:8}^{-1}\Sigma_{2:8,1}=-\Theta_{2:8,1}/\Theta_{11}$ |
| noise variance $\sigma^2$ | $1/\Theta_{11}$ |

Two consequences drive the rest of the module. (1) **To find the neighbours of a node, regress it on all the others and
see which coefficients are non-zero** — with a lasso, to get exact zeros (Section 9). (2) The blue cloud shows that, with
finite data, "non-zero" must be decided statistically.

✏️ **Your turn.** Reduce the sample size in 7.3 to `N = 100`. How large do the non-neighbour coefficients get? Could
you still tell the 16 edge coefficients from the 40 zeros?

### 📘 Lecture notes — inference in a GGM: marginals, conditionals and regression

**Marginals:** delete the rows and columns of the variables you integrate out — $p(x_1)=\mathcal N(\mu_1,\sigma_1^2)$,
$p(x_1,x_2)$ = the $(1,2)$ sub-blocks of $\boldsymbol\mu$ and $\Sigma$.
**Conditionals:** $p(x_1\mid x_2)=p(x_1,x_2)/p(x_2)$ is Gaussian; one variable given all others:

$$p(x_1\mid x_2,\dots,x_8)=\mathcal N\big(\mu_1+(\mathbf x_{2:8}-\boldsymbol\mu_{2:8})^\top\Sigma_{2:8}^{-1}\Sigma_{2:8,1},\ \sigma_1^2-\Sigma_{1,2:8}\Sigma_{2:8}^{-1}\Sigma_{2:8,1}\big).$$

*In words:* the conditional mean is linear in the other variables; the variance shrinks by what they explain.

**Connection to linear regression** $p(y\mid X)=\mathcal N(\beta_0+X\beta,\sigma^2)$: prior mean $\mu_1\leftrightarrow\beta_0$,
other observations $\leftrightarrow X$, their effects $\leftrightarrow\beta$. Completing the square (zero mean):

$$x_i\mid\mathbf x_{-i}\sim\mathcal N\Big(-\sum_{j\neq i}\frac{\Theta_{ij}}{\Theta_{ii}}x_j,\ \frac1{\Theta_{ii}}\Big).$$

*In words:* coefficient $-\Theta_{ij}/\Theta_{ii}$ (0 for non-neighbours), residual variance $1/\Theta_{ii}$. Ring:
$x_1$ depends only on $x_2$ ($-0.529$) and $x_8$ ($-0.176$), residual variance $0.588$.

> 🎓 **Exam tip.** Be able to derive $x_i\mid\mathbf x_{-i}\sim\mathcal N(-\sum_{j\neq i}\Theta_{ij}x_j/\Theta_{ii},\,1/\Theta_{ii})$
(zero mean) by completing the square — it is short, and it explains both the local Markov property and neighbourhood
selection.

---
## 8 · Learning a GGM by maximum likelihood

> 📍 **Where we are.** So far $\Theta$ was given. In practice we have data — $N$ samples (cells, individuals) of $K$
> variables (proteins, genes) — and must *estimate* $\Theta$. This is step 2 of the course's two-step template (model,
> then estimation), done here by maximum likelihood (Module 05). Two questions: what is the MLE of $\Theta$, and what
> should we do if we want the graph to be **sparse**?

The training data form an $N\times K$ matrix with one sample $X^i=[x_{i1},\dots,x_{iK}]$ per row ($K$ variables, which
plays the role of the course's $J$ in the regression modules). We assume the data are **centred** (column means
subtracted), so $X^i\sim\mathcal N(\mathbf 0,\Sigma)$ with $\Theta=\Sigma^{-1}$. Because
$\det(\Sigma)^{-1/2}=\det(\Theta)^{1/2}$, the density of one sample is

$$f(\mathbf x)=(2\pi)^{-K/2}\det(\Sigma)^{-1/2}\exp\big\{-\tfrac12\mathbf x^\top\Sigma^{-1}\mathbf x\big\}
\;\propto\;\det(\Theta)^{1/2}\exp\big\{-\tfrac12\mathbf x^\top\Theta\,\mathbf x\big\}.$$

*In words:* written with the precision matrix, the density is a determinant (which keeps it normalised) times an
exponential of the "energy" $\mathbf x^\top\Theta\mathbf x$.

🧮 **Math refresher — trace.** The **trace** $\operatorname{tr}(M)$ of a square matrix is the sum of its diagonal
entries. Two facts: $\operatorname{tr}(AB)=\operatorname{tr}(BA)$, and a single number is its own trace. Hence
$\mathbf x^\top\Theta\mathbf x=\operatorname{tr}(\mathbf x^\top\Theta\mathbf x)=\operatorname{tr}(\mathbf x\mathbf x^\top\Theta)$
— the "trace trick", which lets us add up the energies of all samples with a single matrix.

The average log-likelihood of the $N$ samples is (dropping the constant $-\tfrac K2\log2\pi$)

$$\ell(\Theta)=\frac1N\sum_{i=1}^N\log f(\mathbf x^i)=\tfrac12\log\det\Theta-\frac{1}{2N}\sum_{i=1}^N(\mathbf x^i)^\top\Theta\,\mathbf x^i
=\tfrac12\log\det\Theta-\tfrac12\operatorname{tr}(S\Theta),\qquad S=\frac1N\sum_{i=1}^N\mathbf x^i(\mathbf x^i)^\top.$$

*In words:* the likelihood rewards a large determinant (a "tight" distribution) and penalises energy on the observed data;
the data enter only through the **sample covariance matrix** $S$.

⚠️ **A factor-of-2 trap.** You will sometimes see the sample covariance written as $S=\frac{1}{2N}\sum_i\mathbf x^i(\mathbf x^i)^\top$
next to this formula. That double-counts the factor $\tfrac12$: with $\tfrac12\operatorname{tr}(S\Theta)$ in the
log-likelihood, $S$ must be $\frac1N\sum_i\mathbf x^i(\mathbf x^i)^\top$ for the equality to hold and for the solution
below to be correct.

**The maximiser.** Maximum likelihood for the GGM is $\arg\min_\Theta\,-\tfrac12\log\det\Theta+\tfrac12\operatorname{tr}(S\Theta)$.
Minimising $-\tfrac12\log\det\Theta+\tfrac12\operatorname{tr}(S\Theta)$ uses two facts
of matrix calculus: the derivative of $\log\det\Theta$ with respect to $\Theta$ is $\Theta^{-1}$, and that of
$\operatorname{tr}(S\Theta)$ is $S$. Setting the derivative to zero:

$$-\tfrac12\Theta^{-1}+\tfrac12S=0\quad\Longrightarrow\quad\hat\Theta_{\text{MLE}}=S^{-1}.$$

*In words:* the best-fitting precision matrix is simply the inverse of the sample covariance matrix.

In [ ]:
# ── 8.1 The GGM log-likelihood, the trace trick, and the maximiser Θ̂ = S⁻¹ ──
rng = np.random.default_rng(SEED + 8)


def ggm_loglik(Theta, S) -> float:
    """Average log-likelihood of centred data up to a constant: ½ log det Θ − ½ tr(SΘ)."""
    sign, logdet = np.linalg.slogdet(Theta)
    return -np.inf if sign <= 0 else 0.5 * logdet - 0.5 * np.trace(S @ Theta)


N = 500
Xc = sample_mvn(np.zeros(8), SIGMA8, N, rng)
Xc = Xc - Xc.mean(axis=0)                                  # centre the data
S8 = Xc.T @ Xc / N                                         # sample covariance with 1/N
print("trace trick: (1/N) Σ xᵢᵀΘxᵢ =", round(np.mean(np.sum((Xc @ THETA8) * Xc, axis=1)), 6),
      "   tr(SΘ) =", round(np.trace(S8 @ THETA8), 6))
Theta_hat = np.linalg.inv(S8)
nudged = []
for _ in range(1000):
    E = rng.normal(0, 0.05, (8, 8))
    nudged.append(ggm_loglik(Theta_hat + (E + E.T) / 2, S8))     # a small symmetric nudge
print(f"ℓ(S⁻¹) = {ggm_loglik(Theta_hat, S8):.4f}   ℓ(true Θ) = {ggm_loglik(THETA8, S8):.4f}"
      f"   best of 1,000 nudged matrices = {max(nudged):.4f}")
print("diagonal of the true Θ        :", np.round(np.diag(THETA8), 2))
print("diagonal of S⁻¹ (S with 1/N)  :", np.round(np.diag(Theta_hat), 2))
print("diagonal if S used 1/(2N)     :", np.round(np.diag(np.linalg.inv(S8 / 2)), 2), " ← twice too large")

🔍 **What just happened?** The trace trick holds to every printed digit. $S^{-1}$ has a higher log-likelihood than any
of 1,000 randomly nudged matrices — and even higher than the *true* $\Theta$, because the MLE fits the particular sample
(including its noise) as well as possible. Using the $\frac{1}{2N}$ definition of $S$ would make every precision
entry twice too large. Now the important question: how good is $S^{-1}$ as an estimate of the *graph*?

In [ ]:
# ── 8.2 How good is the MLE? Dense and noisy for small N, impossible for N ≤ K ──
true_zero = np.isclose(THETA8, 0)
print("     N   max |Θ̂ij| where Θij = 0   mean |error| on edges   exact zeros in Θ̂")
estimates = {}
for N in (20, 50, 200, 1000, 10_000):
    Xn = sample_mvn(np.zeros(8), SIGMA8, N, rng)
    Xn -= Xn.mean(axis=0)
    estimates[N] = np.linalg.inv(Xn.T @ Xn / N)
    err_edges = np.abs(estimates[N] - THETA8)[edge_mask].mean()
    print(f"{N:6d}   {np.abs(estimates[N][true_zero]).max():22.3f}   {err_edges:20.3f}   {int(np.sum(estimates[N] == 0)):12d}")
Xn = sample_mvn(np.zeros(8), SIGMA8, 6, rng)
Xn -= Xn.mean(axis=0)
S_small = Xn.T @ Xn / 6
print(f"\nN = 6 < K = 8: rank of S = {np.linalg.matrix_rank(S_small)} (an inverse needs 8); "
      f"condition number = {np.linalg.cond(S_small):.1e} → S cannot be inverted")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
show_matrix(axes[0], THETA8, NAMES8, "true Θ (8 edges)", fmt="{:.1f}", vmax=3)
show_matrix(axes[1], estimates[50], NAMES8, "MLE Θ̂ = S⁻¹, N = 50", fmt="{:.1f}", vmax=3)
show_matrix(axes[2], estimates[10_000], NAMES8, "MLE Θ̂ = S⁻¹, N = 10,000", fmt="{:.1f}", vmax=3)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Three problems, all visible in the printout and the heat maps:

1. **Dense.** The MLE never contains an exact zero, whatever $N$: every pair would get an edge.
2. **Noisy.** With $N=50$, entries that should be 0 reach magnitudes comparable to, or larger than, the real edge
   weights (0.3–1.0); only with
   thousands of samples do they shrink towards 0 (roughly like $1/\sqrt N$).
3. **Impossible for $N\le K$.** After centring, $S$ has rank at most $N-1$, so with 6 samples of 8 variables it is
   singular and has no inverse. In genomics this is the normal situation: $K\approx20{,}000$ genes measured in a few
   hundred individuals.

Biologists want a network that is **sparse** — a dense network with an edge between almost every pair is hard to
interpret, a sparse one is easy — and one that can be estimated when $N<K$. Both needs point to the same tool you met
in Module 07: the lasso.

### 📘 Lecture notes — learning a GGM by maximum likelihood

**Data:** $N$ centred samples of $K$ variables, $X^i\sim\mathcal N(\mathbf 0,\Theta^{-1})$, with density
$f(\mathbf x)\propto\det(\Theta)^{1/2}\exp\{-\frac12\mathbf x^\top\Theta\mathbf x\}$.

**Log-likelihood** (trace trick $\mathbf x^\top\Theta\mathbf x=\operatorname{tr}(\mathbf x\mathbf x^\top\Theta)$):

$$\ell(\Theta)=\tfrac12\log\det\Theta-\tfrac12\operatorname{tr}(S\Theta),\qquad S=\frac1N\sum_i\mathbf x^i(\mathbf x^i)^\top .$$

*In words:* the data enter only through the sample covariance $S$.

**MLE:** $\arg\min_\Theta-\tfrac12\log\det\Theta+\tfrac12\operatorname{tr}(S\Theta)$; with
$\partial\log\det\Theta/\partial\Theta=\Theta^{-1}$ and $\partial\operatorname{tr}(S\Theta)/\partial\Theta=S$ the
solution is $\Theta^*=S^{-1}$.

**Why it is not enough:** no exact zeros (a dense, hard-to-interpret network), noisy for small $N$, and no inverse
when $N\le K$ (the centred $S$ has rank $\le N-1$). How can we encourage sparsity?

> 🎓 **Exam tip.** Know the three-line derivation ($\ell$ in trace form → derivative → $S^{-1}$) and the two reasons the MLE is not
used for network discovery: no exact zeros, and no inverse when $N\le K$. Watch the factor $\frac1N$ versus $\frac1{2N}$
in $S$: with $\frac1{2N}$ the "solution" would be twice the correct precision matrix.

---
## 9 · Sparsity: the graphical lasso and neighbourhood selection

> 📍 **Where we are.** The MLE gives a dense, noisy network (Section 8). In Module 07 an $L_1$ (lasso) penalty set
> regression coefficients exactly to zero; Section 7 showed that GGM edges *are* regression coefficients. Here we put the
> two together.

To learn a **sparse** GGM, add a lasso penalty to the negative log-likelihood:

$$\hat\Theta=\arg\min_{\Theta}\;-\tfrac12\log\det\Theta+\tfrac12\operatorname{tr}(S\Theta)+\lambda\,\|\Theta\|_1,
\qquad\|\Theta\|_1=\sum_{j\neq k}|\Theta_{jk}|.$$

*In words:* fit the data well (first two terms) but pay $\lambda$ for every unit of absolute edge weight; a large
$\lambda$ makes it cheaper to set weak edges to exactly zero. $\|\Theta\|_1$ here sums only the **off-diagonal** entries
— different from the "$\ell_1$ matrix norm" of linear algebra (the largest absolute column sum).

Three facts about this problem:

- The problem is **convex** — one bowl-shaped valley, so any local minimum is the global one — but it cannot be solved
  by setting the derivative to zero (the absolute value has a kink at 0).
- **Scaling.** Friedman, Hastie & Tibshirani (2008) and scikit-learn multiply the whole objective by 2 and minimise
  $-\log\det\Theta+\operatorname{tr}(S\Theta)+\lambda\|\Theta\|_1$; their $\lambda$ is twice the $\lambda$ of the
  formula above. We use this convention in the code.
- **The diagonal.** Setting the derivative with respect to a diagonal entry to zero gives
  $(\Theta^{-1})_{ii}=S_{ii}+\lambda$ — *if* the penalty also includes the diagonal entries, as in Friedman et al.'s
  original formulation and with their scaling of $\lambda$ (with the factors $\tfrac12$ above it would read
  $S_{ii}+2\lambda$). With the off-diagonal penalty written above (and in scikit-learn) the diagonal of $\Theta^{-1}$
  stays at $S_{ii}$. We check both below.

🧮 **Math refresher — soft-thresholding (Module 07).** The one-variable lasso problem
$\min_\beta\tfrac12a\beta^2-b\beta+\lambda|\beta|$ (with $a>0$) has the solution $\beta=S(b,\lambda)/a$, where

$$S(z,t)=\operatorname{sign}(z)\,\max(|z|-t,\,0).$$

*In words:* move $z$ towards zero by $t$, and if it would cross zero, stop at exactly zero. **Coordinate descent**
applies this to one coefficient at a time, holding the others fixed, and cycles until nothing changes. For the problem
$\min_\beta\tfrac12\beta^\top A\beta-b^\top\beta+\lambda\|\beta\|_1$ the update of coordinate $j$ is
$\beta_j\leftarrow S\big(b_j-\sum_{k\neq j}A_{jk}\beta_k,\;\lambda\big)/A_{jj}$.

**Block coordinate descent** (Friedman et al. 2008; Mazumder & Hastie 2012). The graphical lasso repeatedly cycles
through all nodes and, in each step, optimises only a single row/column of $W$, the working estimate of $\Theta^{-1}$
(the picture after the next cell shows the cycle). Partition $W$, $S$ and $\Theta$ into "everything except node $j$"
(block 11), the column of node $j$ (block 12) and its diagonal entry (block 22):

$$\Theta=\begin{bmatrix}\Theta_{11}&\theta_{12}\\\theta_{12}^\top&\theta_{22}\end{bmatrix},\quad
S=\begin{bmatrix}S_{11}&s_{12}\\s_{12}^\top&s_{22}\end{bmatrix},\quad
W=\begin{bmatrix}W_{11}&w_{12}\\w_{12}^\top&w_{22}\end{bmatrix}.$$

*In words:* block 11 holds all pairs not involving node $j$, block 12 the column of node $j$, block 22 its diagonal entry.
Because $W\Theta=I$, the off-diagonal block of column $j$ gives $W_{11}\theta_{12}+w_{12}\theta_{22}=0$, so
$w_{12}=W_{11}\beta$ with $\beta=-\theta_{12}/\theta_{22}$ (Friedman et al. 2008); the diagonal entry gives
$w_{12}^\top\theta_{12}+w_{22}\theta_{22}=1$, i.e. $\theta_{22}=1/(w_{22}-w_{12}^\top\beta)$. Holding everything except
row/column $j$ fixed, optimising $\theta_{12}$ is then equivalent to a lasso problem in $\beta$:

$$\min_{\beta}\;\tfrac12\big\|W_{11}^{1/2}\beta-W_{11}^{-1/2}s_{12}\big\|_2^2+\lambda\|\beta\|_1
\;=\;\min_\beta\;\tfrac12\beta^\top W_{11}\beta-s_{12}^\top\beta+\lambda\|\beta\|_1+\text{const}.$$

*In words:* expanding the square shows that the inner problem is exactly the $A=W_{11}$, $b=s_{12}$ lasso of the
refresher above; it is like regressing node $j$ on the other nodes, but using the current covariance estimate $W_{11}$
instead of the raw data. After solving it, set $w_{12}=W_{11}\beta$; at the end, read off
$\hat\theta_{22}=1/(w_{22}-w_{12}^\top\beta)$ and $\hat\theta_{12}=-\hat\theta_{22}\beta$.

**Algorithm (block coordinate descent for the graphical lasso).**
*Initialise* $W=S+\lambda I$ (diagonal penalised) or $W=S$ (off-diagonal penalty only) and keep its diagonal fixed.
*Repeat until convergence:* for $j=1,\dots,K$: (i) partition $W$ and $S$ into the four blocks, where the upper-left
block consists of all but the $j$-th row/column; (ii) solve the lasso problem above for $\beta$; (iii) update
$w_{12}=W_{11}\beta$. *Finally* set $\hat\theta_{22}=1/(w_{22}-w_{12}^\top\beta)$ and $\hat\theta_{12}=-\hat\theta_{22}\beta$
for every column.

In [ ]:
# ── 9.1 Soft-thresholding and one lasso solver for both uses ──
rng = np.random.default_rng(SEED + 9)


def soft_threshold(z, t):
    """S(z, t) = sign(z)·max(|z| − t, 0): shrink towards 0 by t; exactly 0 inside [−t, t]."""
    return np.sign(z) * np.maximum(np.abs(z) - t, 0.0)


def lasso_cd(A, b, lam, beta=None, max_sweeps=1000, tol=1e-8):
    """Minimise ½ βᵀAβ − bᵀβ + lam·Σ|βⱼ| by cyclic coordinate descent.

    With A = XᵀX/N and b = Xᵀy/N this is the lasso regression of Module 07 (½·MSE + penalty);
    with A = W₁₁ and b = s₁₂ it is the inner problem of the graphical lasso.
    `beta` is an optional warm start.
    """
    beta = np.zeros(len(b)) if beta is None else beta.copy()
    for _ in range(max_sweeps):
        biggest = 0.0
        for j in range(len(b)):
            partial = b[j] - A[j] @ beta + A[j, j] * beta[j]   # bⱼ − Σ_{k≠j} Aⱼₖ βₖ
            new = soft_threshold(partial, lam) / A[j, j]
            biggest = max(biggest, abs(new - beta[j]))
            beta[j] = new
        if biggest < tol:
            break
    return beta


# check against scikit-learn's Lasso, which minimises (1/2N)‖y − Xβ‖² + α‖β‖₁ — the same objective
Xr = rng.normal(size=(200, 6))
yr = Xr @ np.array([1.5, 0, 0, -1.0, 0, 0.3]) + rng.normal(0, 1, 200)
print("ours        :", np.round(lasso_cd(Xr.T @ Xr / 200, Xr.T @ yr / 200, lam=0.2), 4))
print("scikit-learn:", np.round(Lasso(alpha=0.2, fit_intercept=False, tol=1e-10, max_iter=100_000).fit(Xr, yr).coef_, 4))

In [ ]:
# ── 9.2 The graphical lasso by block coordinate descent ──
def graphical_lasso_bcd(S, lam, penalize_diagonal=False, max_sweeps=200, tol=1e-7):
    """Sparse precision matrix: argmin −log det Θ + tr(SΘ) + lam·Σ_{j≠k}|Θjk| (Friedman et al. 2008).

    W is the working covariance (estimate of Θ⁻¹). Each step updates one row/column of W by solving a lasso
    problem; at the end Θ is read off from W and the lasso coefficients. With penalize_diagonal=True the
    diagonal is penalised too and W starts at S + lam·I, as in Friedman et al.'s algorithm. Returns (Θ, W).
    """
    K = S.shape[0]
    W = S + lam * np.eye(K) if penalize_diagonal else S.copy()
    B = np.zeros((K, K - 1))                               # one β per column, kept as warm starts
    for _ in range(max_sweeps):
        W_old = W.copy()
        for j in range(K):
            rest = np.arange(K) != j
            W11, s12 = W[np.ix_(rest, rest)], S[rest, j]   # (i) partition
            B[j] = lasso_cd(W11, s12, lam, beta=B[j])      # (ii) solve the lasso sub-problem
            W[rest, j] = W[j, rest] = W11 @ B[j]           # (iii) w₁₂ = W₁₁ β
        if np.abs(W - W_old).max() < tol:
            break
    Theta = np.zeros((K, K))
    for j in range(K):                                     # θ₂₂ = 1/(w₂₂ − w₁₂ᵀβ),  θ₁₂ = −θ₂₂ β
        rest = np.arange(K) != j
        Theta[j, j] = 1.0 / (W[j, j] - W[rest, j] @ B[j])
        Theta[rest, j] = -Theta[j, j] * B[j]
    return (Theta + Theta.T) / 2, W


def glasso_objective(Theta, S, lam) -> float:
    """−log det Θ + tr(SΘ) + lam·(sum of |off-diagonal entries|)."""
    off_l1 = np.abs(Theta).sum() - np.abs(np.diag(Theta)).sum()
    return -np.linalg.slogdet(Theta)[1] + np.trace(S @ Theta) + lam * off_l1


lam = 0.1
Th_ours, W_ours = graphical_lasso_bcd(S8, lam)                 # S8: the N = 500 simulated sample of 8.1
W_sk, Th_sk = graphical_lasso(S8, alpha=lam)                # scikit-learn's default tolerance
iu8 = np.triu_indices(8, 1)
print(f"λ = {lam}: max |Θ_ours − Θ_sklearn| = {np.abs(Th_ours - Th_sk).max():.1e}")
print(f"objective: ours {glasso_objective(Th_ours, S8, lam):.6f}   scikit-learn {glasso_objective(Th_sk, S8, lam):.6f}")
print("edges — ours:", int((np.abs(Th_ours[iu8]) > 1e-8).sum()), "  scikit-learn:",
      int((np.abs(Th_sk[iu8]) > 1e-8).sum()), "  true graph: 8   (MLE: 28)")
print("W_ii − S_ii, off-diagonal penalty :", np.round(np.diag(W_ours) - np.diag(S8), 4))
_, W_diag = graphical_lasso_bcd(S8, lam, penalize_diagonal=True)
print("W_ii − S_ii, diagonal penalised   :", np.round(np.diag(W_diag) - np.diag(S8), 4))

# dense versus sparse: the MLE network and the graphical-lasso network of the same 500 samples
fig, axes = plt.subplots(1, 2, figsize=(8.5, 4))
mle_edges = nonzero_edges(np.linalg.inv(S8), NAMES8)
glasso_edges = nonzero_edges(Th_ours, NAMES8, tol=1e-8)
draw_graph(mle_edges, POS8, directed=False, ax=axes[0],
           title=f"dense network (hard to interpret)\nMLE S⁻¹: {len(mle_edges)} edges")
draw_graph(glasso_edges, POS8, directed=False, ax=axes[1],
           title=f"sparse network (easy to interpret)\ngraphical lasso, λ = {lam}: {len(glasso_edges)} edges")
plt.tight_layout(); plt.show()

In [ ]:
# ── 9.2b Picture: block coordinate descent updates one row/column of W at a time ──
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
for ax, j in zip(axes, [0, 1, 2, 7]):
    block = np.zeros((8, 8))
    block[:, j] = block[j, :] = 1                          # w₁₂ (the row/column of node j)
    block[j, j] = 2                                        # w₂₂
    ax.imshow(block, cmap="Blues", vmin=-0.6, vmax=2.2)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"step j = {j + 1}: update column {j + 1}\n(light = W₁₁, blue = w₁₂, dark = w₂₂)", fontsize=8)
fig.suptitle("cycle j = 1, 2, …, K, 1, 2, … until W stops changing (each step = one lasso problem)", fontsize=9)
plt.tight_layout(rect=(0, 0, 1, 0.88)); plt.show()

🔍 **What just happened?** Our from-scratch lasso agrees with scikit-learn's `Lasso` to four decimals, and our
graphical lasso agrees with scikit-learn's `graphical_lasso`: the same objective value (to the printed precision) and the
same set of edges. Unlike the MLE, the result contains exact zeros: the network pictures show the dense MLE network
(an edge for every one of the 28 pairs) next to the sparse graphical-lasso network of the same 500 samples. The diagonal
check settles the formula for the diagonal: $W_{ii}=S_{ii}$ when only off-diagonal entries are penalised, and
$W_{ii}=S_{ii}+\lambda$ when the diagonal is penalised as well. The four small matrices picture one sweep of the
algorithm: in each step, the row and column of one node are re-estimated while the rest of $W$ is held fixed.

### 9.3 Neighbourhood selection — lasso regressions, one node at a time

Section 7 suggests an even simpler method (Meinshausen & Bühlmann 2006): **lasso-regress each
variable on all the others** and connect $j$ and $k$ if the coefficient is non-zero. Each node gets its own regression, so
the two regressions for a pair may disagree; the **AND** rule keeps an edge only if both select it, the **OR** rule if
either does. With standardised data, the regression of node $j$ is our `lasso_cd` with $A=S_{-j,-j}$ (correlations among
the other nodes) and $b=S_{-j,j}$.

To evaluate both methods we need a network whose truth we know, so we SIMULATE one with a biologically flavoured shape:
a six-kinase **cascade** K1 → … → K6 that activates a **transcription factor** TF, which in turn controls five target
genes g1–g5 (a *hub*). Only $N=200$ samples are drawn — a realistic, not generous, sample size.

In [ ]:
# ── 9.3 A SIMULATED sparse network: a kinase cascade feeding a transcription-factor hub ──
NAMES12 = ["K1", "K2", "K3", "K4", "K5", "K6", "TF", "g1", "g2", "g3", "g4", "g5"]
TRUE12 = [("K1", "K2"), ("K2", "K3"), ("K3", "K4"), ("K4", "K5"), ("K5", "K6"), ("K6", "TF"),
          ("TF", "g1"), ("TF", "g2"), ("TF", "g3"), ("TF", "g4"), ("TF", "g5")]
ix12 = {n: k for k, n in enumerate(NAMES12)}
THETA12 = np.eye(12)
for u, v in TRUE12:
    THETA12[ix12[u], ix12[v]] = THETA12[ix12[v], ix12[u]] = -0.4   # Θ < 0 ⇒ positive partial correlation
THETA12[ix12["TF"], ix12["TF"]] = 2.5                            # the hub needs a larger diagonal to stay PD
print("smallest eigenvalue of Θ:", round(np.linalg.eigvalsh(THETA12).min(), 3))
truth_pc = partial_corr(THETA12)
print("true partial correlations: cascade", round(truth_pc[0, 1], 3), "| K6–TF and TF–targets", round(truth_pc[5, 6], 3))
X12 = sample_mvn(np.zeros(12), np.linalg.inv(THETA12), 200, rng)
Z12 = (X12 - X12.mean(axis=0)) / X12.std(axis=0)
S12 = Z12.T @ Z12 / len(Z12)                                   # correlation matrix of the N = 200 samples
truth12 = np.zeros((12, 12), bool)
for u, v in TRUE12:
    truth12[ix12[u], ix12[v]] = truth12[ix12[v], ix12[u]] = True
iu12 = np.triu_indices(12, 1)
POS12 = {**{f"K{k}": (0.02 + 0.2 * (k - 1), 0.98) for k in range(1, 6)}, "K6": (0.97, 0.7), "TF": (0.55, 0.45),
         **{f"g{k}": (0.55 + 0.4 * np.cos(a), 0.45 + 0.4 * np.sin(a))
            for k, a in zip(range(1, 6), np.radians([-165, -125, -90, -55, -15]))}}


def neighbourhood_selection(S, lam):
    """Meinshausen–Bühlmann: lasso-regress every variable on all others (S = correlation matrix).

    Returns the AND-rule and OR-rule adjacency matrices (K × K booleans).
    """
    K = S.shape[0]
    sel = np.zeros((K, K), bool)
    for j in range(K):
        rest = np.arange(K) != j
        sel[j, rest] = np.abs(lasso_cd(S[np.ix_(rest, rest)], S[rest, j], lam)) > 1e-10
    return sel & sel.T, sel | sel.T

In [ ]:
# ── 9.4 Edge recovery along the regularisation path ──
lams = np.geomspace(0.01, 0.5, 15)
curves = {"graphical lasso": [], "neighbourhood selection (AND)": [], "neighbourhood selection (OR)": []}
for lam in lams:
    Th, _ = graphical_lasso_bcd(S12, lam)
    ns_and, ns_or = neighbourhood_selection(S12, lam)
    for name, A in zip(curves, [np.abs(Th) > 1e-8, ns_and, ns_or]):
        p, r, _ = precision_recall_f1(truth12[iu12], A[iu12])
        curves[name].append((p, r, int(A[iu12].sum())))
print("   λ     graphical lasso: edges precision recall  |  NS-AND: edges precision recall")
for lam, g, a in zip(lams, curves["graphical lasso"], curves["neighbourhood selection (AND)"]):
    print(f"{lam:6.3f}   {g[2]:20d} {g[0]:9.2f} {g[1]:6.2f}  | {a[2]:13d} {a[0]:9.2f} {a[1]:6.2f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for name, style in zip(curves, ["o-", "s--", "^:"]):
    arr = np.array(curves[name])
    axes[0].plot(lams, arr[:, 0], style, label=name, ms=4)
    axes[1].plot(lams, arr[:, 1], style, label=name, ms=4)
    axes[2].plot(lams, arr[:, 2], style, label=name, ms=4)
axes[2].axhline(len(TRUE12), color="0.5", ls=":", lw=1)
for ax, lab in zip(axes, ["precision (found edges that are real)", "recall (real edges that are found)", "number of edges"]):
    ax.set_xscale("log"); ax.set_xlabel("penalty λ (log scale)"); ax.set_ylabel(lab)
axes[0].legend(fontsize=7)
axes[1].set_title("SIMULATED cascade + hub, N = 200, 11 true edges of 66 pairs")
plt.tight_layout(); plt.show()

🔍 **What just happened?** As $\lambda$ grows, every method keeps fewer edges: **precision rises** (what remains is
mostly real) while **recall falls** (weaker real edges are dropped). At the smallest $\lambda$ all 11 true edges are found
but buried among about 60 edges (precision below 0.2); at $\lambda=0.5$ nothing is left. In between, each method has a
window in which it recovers the true network exactly — the graphical lasso at $\lambda\approx0.22$, neighbourhood
selection (AND) at $\lambda\approx0.12$–$0.16$ — after which recall drops. The two methods solve different problems, so
their $\lambda$ values are not directly comparable; the OR rule keeps more edges than the AND rule at the same $\lambda$.
There is no free lunch: **$\lambda$ trades false discoveries against missed edges**, and the dotted line (11 edges)
reminds you that in real data the "right" sparsity is unknown.

In [ ]:
# ── 9.5 What the recovered networks look like ──
def draw_estimated(ax, est_edges, true_edges, pos, title, radius=0.07):
    """Correct edges solid (draw_graph), false positives red dashed, missed true edges grey dotted."""
    true_set, est_set = {frozenset(e) for e in true_edges}, {frozenset(e) for e in est_edges}
    for e in true_set - est_set:
        (xa, ya), (xb, yb) = (pos[n] for n in e)
        ax.plot([xa, xb], [ya, yb], ":", color="0.6", lw=1, zorder=1)
    for e in est_set - true_set:
        (xa, ya), (xb, yb) = (pos[n] for n in e)
        ax.plot([xa, xb], [ya, yb], "--", color="tab:red", lw=1.1, zorder=1)
    draw_graph([tuple(e) for e in est_set & true_set], pos, directed=False, ax=ax, title=title, radius=radius)


fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, lam in zip(axes, [0.03, 0.22, 0.4]):
    Th, _ = graphical_lasso_bcd(S12, lam)
    est = nonzero_edges(Th, NAMES12, tol=1e-8)
    p, r, _ = precision_recall_f1(truth12[iu12], (np.abs(Th) > 1e-8)[iu12])
    draw_estimated(ax, est, TRUE12, POS12, f"graphical lasso, λ = {lam}\n{len(est)} edges: precision {p:.2f}, recall {r:.2f}")
fig.suptitle("SIMULATED kinase cascade → TF hub (N = 200): solid = correct, red dashed = false, grey dotted = missed",
             fontsize=9)
plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

🔍 **What just happened?** With a small penalty the cascade and the hub are all there, buried in false edges (red);
at $\lambda=0.22$ the recovered network is exactly the truth; with a large penalty only the strongest links survive —
the five cascade edges (true partial correlation 0.40) and a single hub edge, while K6–TF and four of the five
TF–target edges (true partial correlation 0.25) are lost. The sparse network is far easier to read than the dense
MLE — which is the whole point of encouraging sparsity.

✏️ **Your turn.** (a) Change $N$ from 200 to 1,000 in cell 9.3 and re-run 9.3–9.5: at which $\lambda$ do you now get a
perfect network? (b) Set `THETA12[ix12["TF"], ix12["TF"]] = 1.6` (a weaker-regulated hub, stronger edges) and check that
$\Theta$ is still positive definite. Which edges become easier to find?

**How to choose $\lambda$ in practice?** Common choices: maximise the likelihood of held-out samples (cross-validation,
Module 23; scikit-learn's `GraphicalLassoCV`), an information criterion such as BIC, or keep only edges that are
selected in most bootstrap resamples (*stability selection*, used in Section 10). **Efficient implementations:** the
software packages QUIC and BIG&QUIC use Newton's method for the optimisation, with clever approximations; they scale to
tens of thousands up to a million variables and are more efficient than neighbourhood selection.

### 📘 Lecture notes — sparse GGMs and the graphical lasso

**Sparse GGM:** $\hat\Theta=\arg\min_\Theta-\tfrac12\log\det\Theta+\tfrac12\operatorname{tr}(S\Theta)+\lambda\|\Theta\|_1$,
$\|\Theta\|_1=\sum_{j\neq k}|\Theta_{jk}|$ (not the $\ell_1$ matrix norm). *In words:* pay $\lambda$ per unit of edge
weight. Convex, but not solvable by setting the derivative to zero; the diagonal satisfies
$(\Theta^{-1})_{ii}=S_{ii}+\lambda$ when it is penalised too (Friedman et al.'s scaling).

**Graphical lasso (block coordinate descent):** $W$ = working version of $\Theta^{-1}$, initialised at $S+\lambda I$
when the diagonal is penalised (at $S$ with the off-diagonal penalty used in our code); with $\beta=-\theta_{12}/\theta_{22}$, $W\Theta=I$ gives $w_{12}=W_{11}\beta$.
Repeatedly cycle through the nodes; for node $j$ partition $W$, $S$ (11 = all but $j$, 12 = column $j$), solve

$$\min_\beta\ \tfrac12\big\|W_{11}^{1/2}\beta-W_{11}^{-1/2}s_{12}\big\|_2^2+\lambda\|\beta\|_1\quad(=\tfrac12\beta^\top W_{11}\beta-s_{12}^\top\beta+\lambda\|\beta\|_1+\text{const}),$$

*In words:* a lasso regression of node $j$ on the others, using the current covariance estimate instead of data;
then update $w_{12}=W_{11}\beta$; finally $\hat\theta_{22}=1/(w_{22}-w_{12}^\top\beta)$, $\hat\theta_{12}=-\hat\theta_{22}\beta$.
Each inner problem is a lasso, solved by coordinate descent with soft-thresholding $S(z,t)=\operatorname{sign}(z)\max(|z|-t,0)$.

**Alternatives:** neighbourhood selection (a lasso regression per node; AND/OR rule); QUIC and BIG&QUIC (Newton's
method, more efficient than neighbourhood selection).

> 🎓 **Exam tip.** Know *why* the inner problem is a lasso (expand $\tfrac12\|W_{11}^{1/2}\beta-W_{11}^{-1/2}s_{12}\|^2$) and
how $\Theta$ is recovered at the end. As $\lambda$ grows the network gets sparser; precision typically rises and recall
falls. Mind the factor-2 difference in $\lambda$ between the $\tfrac12$-scaled objective and scikit-learn's.

---
## 10 · A T-cell signalling network (Sachs et al. 2005), SIMULATED with a known truth

> 📍 **Where we are.** We have a validated graphical lasso and neighbourhood selection. Now we apply them to single-cell
> data shaped like the study of the biological question and compare the result with the network that generated the
> data — the 20-edge consensus network that Module 17 also used.

**The original study.** Sachs et al. (2005) measured 11 phosphorylated proteins and phospholipids (raf, mek, plc, pip2,
pip3, erk, akt, pka, pkc, p38, jnk) by flow cytometry in thousands of human CD4⁺ T cells, with and without targeted
inhibitors or activators, and learned a network that recovered most known signalling links; the data became a standard
test case for the graphical lasso (Friedman et al. 2008). Real intensities are skewed and partly floored, signalling is
saturating, and the "true" network is only a literature consensus.

**Our data.** `simulate_signaling_cytometry` generates 2,603 unperturbed cells from a **linear-Gaussian model on exactly
the 20 consensus arrows** (each molecule = weighted sum of its parents + Gaussian noise). Here the consensus network
*is* the truth, the data *are* Gaussian, and every learned edge can be checked.

In [ ]:
# ── 10.1 Generate the signalling data (SIMULATED, shaped like Sachs et al. 2005) ──
rng = np.random.default_rng(SEED + 10)
sachs = simulate_signaling_cytometry("continuous")         # unperturbed cells only (observational_only=True)
print("source:", sachs["source"])
D = sachs["data"]
PROT = list(D.columns)
print(D.shape[0], "cells ×", D.shape[1], "molecules:", ", ".join(PROT))
print(D.describe().loc[["mean", "std", "min", "max"]].round(2).to_string())
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, p in zip(axes[:3], ["raf", "pka", "pip3"]):
    ax.hist(D[p], bins=50, color="tab:blue", alpha=0.8)
    ax.set_title(f"SIMULATED {p}: skewness {stats.skew(D[p]):+.2f}"); ax.set_xlabel("log intensity"); ax.set_ylabel("cells")
axes[3].scatter(D["raf"], D["mek"], s=2, alpha=0.3, color="tab:blue")
axes[3].set_title(f"SIMULATED raf vs mek: r = {np.corrcoef(D['raf'], D['mek'])[0, 1]:.2f}")
axes[3].set_xlabel("raf (log intensity)"); axes[3].set_ylabel("mek (log intensity)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The source line says it: these cells are simulated. The distributions are bell-shaped
with skewness close to 0 — by construction, since the model is linear-Gaussian (real cytometry data are not so kind; see
the 🚀 Bonus on the nonparanormal transformation). raf and mek, a kinase and its direct substrate, are strongly
correlated ($N=2{,}603\gg K=11$, so even $S^{-1}$ exists). Before learning anything, let us work out what the *right
answer* is.

### 10.2 The true undirected graph of a Bayesian network: the moral graph

The data were generated by a **linear-Gaussian Bayesian network**: for every molecule $c$ with parents
$\mathrm{pa}(c)$, $x_c=\sum_{p\in\mathrm{pa}(c)}w_{pc}\,x_p+\varepsilon_c$ with independent noise
$\varepsilon_c\sim\mathcal N(0,\omega_c)$ (means dropped). Collect the weights in a matrix $B$ (row = child, column =
parent): $\mathbf x=B\mathbf x+\boldsymbol\varepsilon$, so $(I-B)\mathbf x=\boldsymbol\varepsilon$ and

$$\Theta=\Sigma^{-1}=(I-B)^\top\,\Omega^{-1}\,(I-B),\qquad
\Theta_{ij}=-\frac{w_{ij}}{\omega_j}-\frac{w_{ji}}{\omega_i}+\sum_{c\,:\,i,j\in\mathrm{pa}(c)}\frac{w_{ic}\,w_{jc}}{\omega_c}\quad(i\neq j),$$

where $\Omega$ is the diagonal matrix of noise variances.

*In words:* $\Theta_{ij}$ can be non-zero for two reasons — an arrow between $i$ and $j$ (either direction), or a
**shared child** $c$. So the undirected graph of a Gaussian Bayesian network is its **moral graph**: drop the arrow
heads and "marry" every pair of parents that share a child. This is the Markov-blanket idea of Module 16 in disguise:
in the moral graph, the neighbours of a node are exactly its Bayesian-network Markov blanket (parents, children,
co-parents). A GGM learned from these cells should find the 22 edges of the moral graph, not just the 20 arrows.

In [ ]:
# ── 10.2 The true network: from the 20 arrows to the moral graph, and the true precision matrix ──
parents = {v: [p for p, c in SACHS_CONSENSUS_EDGES if c == v] for v in PROT}
skeleton = {frozenset(e) for e in SACHS_CONSENSUS_EDGES}
coparents = {frozenset(pair) for v in PROT for pair in itertools.combinations(parents[v], 2)} - skeleton
moral = skeleton | coparents
print(f"{len(skeleton)} arrows; co-parent pairs without an arrow: {sorted(tuple(sorted(e)) for e in coparents)}"
      f" → moral graph with {len(moral)} edges")

idx = {p: k for k, p in enumerate(PROT)}
B = np.zeros((11, 11))
for name, w in sachs["weights"].items():                   # true edge weights, e.g. "raf→mek" → 0.75
    p, c = name.split("→")
    B[idx[c], idx[p]] = w                                  # row = child, column = parent
Xc11 = D.to_numpy(float) - D.to_numpy(float).mean(axis=0)
omega = np.array([np.var(Xc11[:, k] - Xc11 @ B[k]) for k in range(11)])   # noise variance: residual given the parents
IB = np.eye(11) - B
THETA_TRUE = IB.T @ np.diag(1 / omega) @ IB
P_TRUE = partial_corr(THETA_TRUE)
iu11 = np.triu_indices(11, 1)
nonzero = {frozenset((PROT[a], PROT[b])) for a, b in zip(*iu11) if abs(THETA_TRUE[a, b]) > 1e-12}
print("non-zero pattern of the true Θ equals the moral graph?", nonzero == moral)
for u, v in [("raf", "mek"), ("pip3", "erk"), ("pip3", "pka"), ("pka", "pkc"), ("raf", "pkc")]:
    arrow = (u, v) if (u, v) in SACHS_CONSENSUS_EDGES else (v, u)
    kind = (f"arrow {arrow[0]}→{arrow[1]}, weight {sachs['weights'][f'{arrow[0]}→{arrow[1]}']:+.2f}"
            if frozenset((u, v)) in skeleton else "co-parents, no arrow")
    print(f"   {u:>4s}–{v:<4s} true partial correlation {P_TRUE[idx[u], idx[v]]:+.3f}   ({kind})")

ring = ["raf", "mek", "erk", "akt", "pip3", "pip2", "plc", "pkc", "jnk", "p38", "pka"]   # a circular layout
POS11 = {n: (0.5 + 0.46 * np.cos(a), 0.5 + 0.46 * np.sin(a))
         for n, a in zip(ring, np.pi / 2 - np.arange(11) * 2 * np.pi / 11)}
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))
draw_graph(list(SACHS_CONSENSUS_EDGES), POS11, directed=True, ax=axes[0], radius=0.06,
           title="the generating network: 20 arrows\n(consensus network of Sachs et al. 2005)")
draw_graph([tuple(e) for e in skeleton], POS11, directed=False, ax=axes[1], radius=0.06,
           title="its moral graph: 20 + 2 edges\n(red dashed = parents of akt, 'married')")
for e in coparents:
    (xa, ya), (xb, yb) = (POS11[n] for n in e)
    axes[1].plot([xa, xb], [ya, yb], "--", color="tab:red", lw=1.5, zorder=1)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Only akt has two parents that are not already linked (pip3 and erk; pip3 and pka), so the
moral graph adds exactly two edges, and the non-zero pattern of the true $\Theta$ is exactly this 22-edge graph. (The
noise variances $\omega_c$ are not returned by the simulator; we estimated them as the residual variance of each molecule
given its *known* parents and weights — learning a Bayesian network with a known graph, Module 17.)

The true partial correlations hold three lessons:

- **Co-parent edges are real GGM edges.** pip3–erk and pip3–pka have clearly non-zero partial correlations (about
  −0.19 and −0.16) although no arrow joins them: conditioning on their common child akt makes them dependent
  ("explaining away", Module 16).
- **Signs can flip.** pkc → pka is a positive arrow, but the partial correlation of pka and pkc is *negative* (about
  −0.31): the pair shares four children (raf, mek, p38, jnk), and conditioning on them pulls the other way.
- **Edges can almost vanish.** pkc → raf is a sizeable arrow, yet its partial correlation is only about +0.03: the
  direct link and the "explaining-away" link through their common child mek nearly cancel — a near-violation of the
  *faithfulness* assumption of Section 6.2.

In [ ]:
# ── 10.3 Correlation versus partial correlation for the 11 signalling molecules ──
Zs, _, _ = standardize(D.to_numpy(float))                 # z-scores, so S is the correlation matrix
S_sachs = Zs.T @ Zs / len(Zs)
P_sachs = partial_corr(np.linalg.inv(S_sachs))            # MLE-based partial correlations (N ≫ K)
truth_mat = np.zeros((11, 11), bool)
for e in moral:
    a, b = (idx[n] for n in e)
    truth_mat[a, b] = truth_mat[b, a] = True
truth11 = truth_mat[iu11]
print(f"true (moral) network: {truth11.sum()} edges among {len(truth11)} pairs")
for u, v in [("raf", "mek"), ("mek", "erk"), ("raf", "erk"), ("erk", "akt"), ("akt", "pka"), ("pkc", "jnk"),
             ("p38", "jnk"), ("pip3", "erk")]:
    a, b = idx[u], idx[v]
    print(f"{u:>4s}–{v:<4s} correlation {S_sachs[a, b]:+.3f}   partial correlation {P_sachs[a, b]:+.3f}"
          f"   (true {P_TRUE[a, b]:+.3f})   true edge? {truth_mat[a, b]}")
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), gridspec_kw={"width_ratios": [1, 1, 0.8]})
for ax, M, t in [(axes[0], S_sachs, "correlation"), (axes[1], P_sachs, "partial correlation (from S⁻¹)")]:
    show_matrix(ax, M, PROT, f"SIMULATED cells: {t}\n(black dots = true edges)", vmax=1)
    rr, cc_ = np.nonzero(truth_mat)
    ax.scatter(cc_, rr + 0.33, s=5, color="k")
axes[2].scatter(P_TRUE[iu11][~truth11], P_sachs[iu11][~truth11], s=14, color="tab:blue", label="no edge (true 0)")
axes[2].scatter(P_TRUE[iu11][truth11], P_sachs[iu11][truth11], s=14, color="tab:red", label="true edges")
axes[2].plot([-0.7, 0.7], [-0.7, 0.7], "k--", lw=1)
axes[2].set_xlabel("true partial correlation"); axes[2].set_ylabel("sample partial correlation (N = 2,603)")
axes[2].set_title("55 pairs: estimate vs truth"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The correlation matrix is a sea of large values: in a signalling cascade almost everything
is correlated with everything (raf–erk +0.80, p38–jnk +0.62 — neither pair is linked). The partial-correlation matrix is
nearly sparse, its large entries sit on the black dots, and raf–erk (linked only *through* mek) and p38–jnk (siblings
with the same parents) keep almost no partial correlation. With 2,603 cells the sample partial correlations fall on the
true values (right panel), and the 33 non-edges cluster near 0 (sampling noise ≈ $\pm1/\sqrt{2{,}603}\approx\pm0.02$).
Can our sparse learners find the 22 edges?

In [ ]:
# ── 10.4 Learn the network: graphical lasso and neighbourhood selection, scored against the truth ──
rows = []
lams_sachs = np.geomspace(0.005, 0.5, 14)
for lam in lams_sachs:
    Th, _ = graphical_lasso_bcd(S_sachs, lam)
    ns_and, _ = neighbourhood_selection(S_sachs, lam)
    for method, A in [("graphical lasso", (np.abs(Th) > 1e-8)[iu11]), ("NS (AND)", ns_and[iu11])]:
        p, r, f1 = precision_recall_f1(truth11, A)
        rows.append((method, lam, int(A.sum()), int((A & truth11).sum()), p, r, f1))
path = pd.DataFrame(rows, columns=["method", "lambda", "edges", "correct", "precision", "recall", "F1"])
print(path.pivot(index="lambda", columns="method", values=["edges", "correct"]).round(3).to_string())
Th_big, _ = graphical_lasso_bcd(S_sachs, lams_sachs[-1])
print(f"graphical lasso at λ = {lams_sachs[-1]}: false edges (marginal correlation) "
      f"{[f'{PROT[a]}–{PROT[b]} ({S_sachs[a, b]:+.2f})' for a, b in zip(*iu11) if abs(Th_big[a, b]) > 1e-8 and not truth_mat[a, b]]}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for (method, grp), style in zip(path.groupby("method", sort=False), ["o-", "s--"]):
    axes[0].plot(grp["lambda"], grp["precision"], style, color="tab:blue", ms=4, label=f"{method}: precision")
    axes[0].plot(grp["lambda"], grp["recall"], style, color="tab:orange", ms=4, label=f"{method}: recall")
axes[0].axhline(22 / 55, color="0.5", ls=":", lw=1, label="precision of random edges (22/55)")
axes[0].set_xscale("log"); axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("penalty λ (log scale)"); axes[0].set_ylabel("score vs the true network")
axes[0].set_title("SIMULATED cells: the two λ paths"); axes[0].legend(fontsize=6.5, loc="lower left")
hits = {name: np.cumsum(truth11[np.argsort(-np.abs(M[iu11]))])           # true edges among the top-k pairs
        for name, M in [("rank by |correlation|", S_sachs), ("rank by |partial correlation|", P_sachs)]}
k = np.arange(1, 56)
for (name, h), style in zip(hits.items(), ["-", "--"]):
    axes[1].plot(k, h, style, label=name)
axes[1].plot(k, k * 22 / 55, ":", color="0.5", label="random ranking (expected)")
axes[1].plot(k, np.minimum(k, 22), color="0.8", lw=4, zorder=0, label="perfect ranking")
axes[1].set_xlabel("top-k pairs"); axes[1].set_ylabel("true edges among them")
axes[1].set_title("which ranking finds the true edges first?"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened? (A result worth understanding.)** The two learners behave very differently.
**Neighbourhood selection** traces a textbook path: at large $\lambda$ only true edges (precision 1.0); as $\lambda$
decreases the weaker true edges join one by one; around $\lambda\approx0.02$–$0.03$ it returns 21 edges, *all* true
(the one it misses, pka–raf, is discussed below); false edges appear only at the smallest $\lambda$. **The graphical
lasso** never gets below about 27 edges, keeps around ten false edges even at its sparsest, and loses recall.

Why? The molecules are strongly correlated (up to 0.91). The $L_1$ penalty shrinks *every* edge, the strong true ones
included; to still reproduce the large observed correlations, the penalised solution adds edges between strongly
correlated non-neighbours (printed for the largest $\lambda$: marginal correlations 0.6–0.8). For large $\lambda$ the
graphical lasso behaves like *thresholding the correlation matrix* — exactly what a GGM should avoid. Both methods
come with guarantees only under "incoherence" conditions — roughly, non-neighbours must not be too strongly correlated
with a node's neighbours (Meinshausen & Bühlmann 2006; Ravikumar et al. 2011); the conditions differ, and this
strongly correlated cascade likely violates the graphical lasso's. With $N\gg K$ the plain MLE works too: ranking pairs by
|partial correlation| (right panel) puts 21 true edges in the top 21, while ranking by |correlation| is perfect only for
the first seven pairs (about 15 true edges among the top 25).

Lesson: sparse estimators are tools for $N\lesssim K$ and for interpretability, not guarantees — check how a method
behaves on simulated data that look like yours. (With real cytometry data one should expect weaker agreement for
every method: real data violate the linear-Gaussian model, and a literature consensus is not the truth.)

With $N\gg K$ we can also **test** each partial correlation. Under "no edge", the Fisher-transformed sample partial
correlation is approximately standard normal:

$$z_{ij}=\operatorname{artanh}(\hat\rho_{ij\mid\text{rest}})\,\sqrt{N-(K-2)-3}\;\sim\;\mathcal N(0,1),$$

*In words:* the transformation $\operatorname{artanh}(r)=\tfrac12\log\frac{1+r}{1-r}$ makes the sampling distribution of
a correlation nearly normal with standard error $1/\sqrt{N-|C|-3}$, where $|C|=K-2$ variables are conditioned on.

In [ ]:
# ── 10.5 With N ≫ K: test every partial correlation (Fisher z, Bonferroni over 55 pairs) ──
N_cells = len(Zs)
z_fisher = np.arctanh(P_sachs[iu11]) * np.sqrt(N_cells - (11 - 2) - 3)
p_fisher = 2 * stats.norm.sf(np.abs(z_fisher))
A_test = p_fisher < 0.05 / len(p_fisher)
pr_test = precision_recall_f1(truth11, A_test)
pair_names = [f"{PROT[a]}–{PROT[b]}" for a, b in zip(*iu11)]
print(f"edges with Bonferroni-adjusted p < 0.05: {A_test.sum()}; true: {(A_test & truth11).sum()} of 22 "
      f"(precision {pr_test[0]:.2f}, recall {pr_test[1]:.2f})")
print("missed true edges:", [n for n, a, t in zip(pair_names, A_test, truth11) if t and not a])
print("false edges      :", [n for n, a, t in zip(pair_names, A_test, truth11) if a and not t])
for u, v in [("pka", "raf"), ("raf", "pkc")]:
    print(f"{u}–{v}: sample partial correlation {P_sachs[idx[u], idx[v]]:+.3f}, marginal correlation {S_sachs[idx[u], idx[v]]:+.3f}")

🔍 **What just happened?** The test keeps 21 pairs, all of them true edges; the only miss is raf–pkc, whose true
partial correlation (+0.03) is too close to zero to detect even with 2,603 cells — the near-cancellation of Section 10.2.
Neighbourhood selection, in contrast, *kept* raf–pkc and missed a different edge, pka–raf. Its partial correlation
(−0.15, printed above) is far from zero, but it has the **opposite sign** of the marginal correlation (+0.75): conditioning on the
shared children reverses the sign. The lasso is known to struggle with such "suppressor" coefficients, whose sign
differs from the marginal association, and shrinks them away first; raf–pkc, with a large marginal correlation (+0.68),
survives. Different methods make different, explainable mistakes — which we can only see because the truth is known.
Now let us look at the learned networks.

In [ ]:
# ── 10.6 Draw the true network and the two learned networks (each at its best-F1 λ) ──
best_lam = {m: path[path["method"] == m].sort_values(["F1", "lambda"], ascending=[False, False]).iloc[0]["lambda"]
            for m in ["graphical lasso", "NS (AND)"]}
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6))
draw_graph([tuple(e) for e in moral], POS11, directed=False, ax=axes[0], radius=0.06,
           title="true network (moral graph)\n22 edges")
for ax, method in zip(axes[1:], ["graphical lasso", "NS (AND)"]):
    best = path[path["method"] == method].sort_values(["F1", "lambda"], ascending=[False, False]).iloc[0]
    lam_b = best["lambda"]
    if method == "graphical lasso":
        A = np.abs(graphical_lasso_bcd(S_sachs, lam_b)[0]) > 1e-8
    else:
        A = neighbourhood_selection(S_sachs, lam_b)[0]
    est = [(PROT[a], PROT[b]) for a, b in zip(*iu11) if A[a, b]]
    draw_estimated(ax, est, [tuple(e) for e in moral], POS11,
                   f"{method}, λ = {lam_b:.3f}: {len(est)} edges\nprecision {best['precision']:.2f}, recall {best['recall']:.2f}",
                   radius=0.06)
    print(f"{method:16s} λ = {lam_b:.3f}: false edges {[f'{u}–{v}' for u, v in est if not truth_mat[idx[u], idx[v]]]}")
fig.suptitle("SIMULATED signalling cells: solid = correct, red dashed = false, grey dotted = missed", fontsize=9)
plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

🔍 **What just happened?** At its best $\lambda$ (highest F1 along the path), neighbourhood selection draws the true
network with a single grey dotted line (pka–raf). The graphical lasso's best $\lambda$ is the smallest on the path: it
is almost the unpenalised MLE, recovers all 22 true edges and keeps 13 weak false edges; a larger $\lambda$ only trades
them for the correlation-driven false edges of 10.4. And *no* undirected method can give the arrows of the generating
network.

### 10.7 What an unmeasured molecule does

Real panels never measure everything. What if a molecule exists in the cell but is **not measured**? Marginalising a
variable out of a Gaussian keeps the rest Gaussian, but changes their precision matrix: all neighbours of the hidden
variable become connected. We delete a column and re-learn.

In [ ]:
# ── 10.7 Hide one molecule (pretend it was not measured) and re-learn the network ──
lam_ns = best_lam["NS (AND)"]                               # the NS (AND) penalty that worked best on the full panel
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
for ax, hidden in zip(axes, ["pka", "mek"]):
    h = idx[hidden]
    keep = [k for k in range(11) if k != h]
    names = [PROT[k] for k in keep]
    # theory: the precision matrix of the measured molecules is a Schur complement — neighbours of h get connected
    Theta_marg = THETA_TRUE[np.ix_(keep, keep)] - np.outer(THETA_TRUE[keep, h], THETA_TRUE[h, keep]) / THETA_TRUE[h, h]
    predicted = {frozenset((names[a], names[b])) for a, b in zip(*np.triu_indices(10, 1))
                 if abs(Theta_marg[a, b]) > 1e-12} - moral
    A_h, _ = neighbourhood_selection(S_sachs[np.ix_(keep, keep)], lam_ns)
    est = [(names[a], names[b]) for a, b in zip(*np.triu_indices(10, 1)) if A_h[a, b]]
    true_kept = [tuple(e) for e in moral if hidden not in e]
    spurious = [(u, v) for u, v in est if frozenset((u, v)) not in moral]
    print(f"{hidden} hidden: theory predicts {len(predicted)} new edges (all pairs of its neighbours not yet linked); "
          f"the learner finds {len(est)} edges, {len(spurious)} not in the full network:")
    print("    predicted and found:", [f"{u}–{v}" for u, v in spurious if frozenset((u, v)) in predicted])
    print("    found, not predicted:", [f"{u}–{v}" for u, v in spurious if frozenset((u, v)) not in predicted])
    pos = {n: POS11[n] for n in names}
    draw_estimated(ax, est, true_kept, pos, f"{hidden} not measured: {len(spurious)} extra edges (red)", radius=0.06)
    ax.text(*POS11[hidden], f"({hidden})", ha="center", va="center", fontsize=9, color="0.5")
fig.suptitle(f"SIMULATED cells, neighbourhood selection (AND), λ = {lam_ns:.3f}", fontsize=9)
plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

🔍 **What just happened?** Theory first: removing variable $h$ replaces the precision matrix of the rest by the *Schur
complement* $\Theta_{rr}-\boldsymbol\theta_{rh}\boldsymbol\theta_{hr}/\Theta_{hh}$, which adds a term to every pair of
neighbours of $h$. Hide **pka** — a kinase acting on six of the other molecules — and the theory predicts 19 new edges
among its eight neighbours; the learner finds 8 of them (printed) plus one weak false positive (pip2–p38). p38 and jnk,
for example, are now linked because both respond to the unmeasured pka: a **latent confounder**. Hide **mek**, the
middle of the raf → mek → erk cascade, and raf and erk become directly linked (plus a weak false positive, raf–jnk): a
**hidden mediator** turns an indirect effect into an apparent direct one. These edges are true partial correlations
*among the measured molecules* — read as direct interactions they are wrong. In real cells many regulators are
unmeasured, one of the main reasons network edges must be validated by experiment.

In [ ]:
# ── 10.8 How stable is each edge? Re-learn the network on 30 bootstrap resamples of the cells ──
n_boot = 30
lam_gl = best_lam["graphical lasso"]
freq = {"graphical lasso": np.zeros((11, 11)), "NS (AND)": np.zeros((11, 11))}
for _ in range(n_boot):
    pick = rng.integers(0, len(Zs), len(Zs))               # resample cells with replacement
    Zb = (Zs[pick] - Zs[pick].mean(axis=0)) / Zs[pick].std(axis=0)
    Sb = Zb.T @ Zb / len(Zb)
    freq["graphical lasso"] += np.abs(graphical_lasso_bcd(Sb, lam_gl)[0]) > 1e-8
    freq["NS (AND)"] += neighbourhood_selection(Sb, lam_ns)[0]
fig, axes = plt.subplots(2, 1, figsize=(11, 5.6))
for ax, (method, F) in zip(axes, freq.items()):
    f = F[iu11] / n_boot
    top = np.argsort(-f, kind="stable")[:34]
    ax.bar(range(34), f[top], color=["tab:blue" if truth11[t] else "tab:orange" for t in top])
    ax.set_xticks(range(34)); ax.set_xticklabels([pair_names[t] for t in top], rotation=70, ha="right", fontsize=6.5)
    lam_m = lam_gl if method == "graphical lasso" else lam_ns
    ax.set_ylabel("selection frequency")
    ax.set_title(f"{method}, λ = {lam_m:.3f}: blue = true edge, orange = false edge", fontsize=9)
    stable = f >= 0.9
    print(f"{method:16s}: {stable.sum()} edges in ≥ 90 % of {n_boot} resamples, of which {(stable & ~truth11).sum()} false")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With 2,603 cells, every true edge a method finds is selected in every resample
(neighbourhood selection never selects pka–raf, its systematic miss), and neighbourhood selection's false candidates
flicker (the most frequent appears in fewer than a quarter of the resamples). But **five false edges of the graphical
lasso are selected in at least 90 % of the resamples** — as stable as the true ones. Stability measures *sampling*
variability, not *bias*: a systematically wrong method is wrong in every resample. Stability is reassuring but is **not**
validation; it buys a short, reproducible list of hypotheses to test — exactly what Sachs et al. did with targeted
perturbations to establish directions and mechanisms.

⚠️ **How to read a GGM edge.** An edge means: *these two quantities are partially correlated — linearly associated after
adjusting for all other measured variables — in this population of samples.* It does **not** mean binding,
phosphorylation, regulation or a direction of influence. It can be created by unmeasured common causes or mediators
(10.7), by married co-parents (10.2), by the estimator itself (10.4) and by mixtures of cell types or states; it can be
missed when effects cancel, when the relation is non-linear, or when the link is inactive in the sampled conditions.

✏️ **Your turn.** (a) Generate all 7,466 cells with `simulate_signaling_cytometry("continuous", observational_only=False)`
— unperturbed cells pooled with cells treated by inhibitors or activators, the setting of the classic graphical-lasso
analysis — and repeat 10.3–10.5. Which edges change? Why might pooling perturbation experiments violate the assumption
that all samples come from **one** Gaussian, and why does an intervention cut the arrows *into* the perturbed molecule?
(b) Hide `pkc` instead of pka in 10.7: which extra edges does the true graph predict?

---
## 11 · Summary: why Gaussian makes undirected models efficient

> 📍 **Where we are.** The whole lecture topic fits in one sentence: *GGMs are undirected probabilistic graphical
> models of continuous variables, and because they are Gaussian, inference and learning are efficient — which is not
> true for undirected models of discrete variables.* Here is the comparison collected from Modules 15–18.

| | Bayesian network (Modules 15–17) | Discrete MRF | Gaussian graphical model |
|---|---|---|---|
| graph | directed, acyclic | undirected, loops allowed | undirected, loops allowed |
| distribution | $\prod_i P(x_i\mid\text{parents})$ | $\frac1Z\prod_C\psi_C(\mathbf x_C)$ | $\mathcal N(\boldsymbol\mu,\Theta^{-1})$ |
| missing edge means | d-separation statements | separation ⇒ independence | $\Theta_{ij}=0$ ⇔ $x_i\perp x_j\mid\text{rest}$ |
| Markov blanket | parents, children, co-parents | neighbours | neighbours |
| normalising constant | 1 (automatic) | sum over all configurations | $(2\pi)^{K/2}\det(\Theta)^{-1/2}$ |
| inference | exact for small/tree graphs, else hard | hard in general | marginals: pick entries; conditionals: regression |
| learning (known graph) | counting | iterative, needs $Z$ | $\hat\Theta=S^{-1}$ (or graphical lasso) |
| structure learning | combinatorial search | hard | convex graphical lasso, neighbourhood selection |

The two families meet in the **moral graph** (Section 10.2): a Gaussian Bayesian network is also a GGM, whose graph is
the DAG with arrow heads dropped and co-parents married. The course's second recurring idea — **conditional
independence buys tractability** — appears here as a sparse precision matrix: each zero removes a direct dependence,
makes the network interpretable and lets us estimate it from fewer samples than an unrestricted covariance would need.

### 📘 Lecture notes — Gaussian graphical models — what you should know

- GGMs are **undirected probabilistic graphical models of continuous variables**: $\mathcal N(\boldsymbol\mu,\Theta^{-1})$,
  edge ⇔ $\Theta_{ij}\neq0$ ⇔ dependence given all other variables; Markov blanket = neighbours.
- **Gaussian ⇒ efficient.** Inference: marginals are sub-blocks, conditionals are linear regressions. Learning:
  $\Theta^*=S^{-1}$, or the convex graphical lasso for sparse networks. The normalising constant is one determinant.
- **Not true for discrete undirected models**, whose normalising constant sums over exponentially many configurations.

> 🎓 **Exam tip.** If asked "why GGMs and not discrete MRFs for expression data?", answer with the normalising constant: one
determinant versus a sum over exponentially many configurations. If asked what a GGM edge means, say "partial
correlation given the measured variables" — not regulation, not direction.

## 📌 Key takeaways

- **Undirected graphical models (MRFs)** write a distribution as a normalised product of clique potentials; they allow
  loops (feedback) and need no edge directions. The Markov blanket of a node is just its neighbours, and graph
  separation gives the pairwise, local and global Markov properties.
- **Discrete MRFs are expensive** because the normalising constant sums over exponentially many configurations;
  **Gaussian MRFs (GGMs) are tractable**: the constant is one determinant.
- In a GGM, $p(\mathbf x)=\mathcal N(\boldsymbol\mu,\Theta^{-1})$ and **$\Theta_{ij}=0\iff x_i\perp x_j\mid$ all other
  variables**. The covariance matrix only reveals connected components; the **precision matrix** reveals the edges.
  Partial correlation $=-\Theta_{ij}/\sqrt{\Theta_{ii}\Theta_{jj}}$ (mind the sign).
- **Inference is easy:** marginals are sub-blocks of $\boldsymbol\mu,\Sigma$; the conditional of one variable given the
  rest is a **linear regression** with coefficients $-\Theta_{ij}/\Theta_{ii}$ and residual variance $1/\Theta_{ii}$.
- **The MLE is $\hat\Theta=S^{-1}$** (with $S=\frac1N\sum\mathbf x\mathbf x^\top$, not $\frac1{2N}$).
  It is dense, noisy, and does not exist when $N\le K$.
- **The graphical lasso** adds $\lambda\sum_{j\ne k}|\Theta_{jk}|$, is convex, is solved by block coordinate descent
  with a lasso in each step, and gives sparse networks; **neighbourhood selection** (node-wise lasso regressions) is a
  simple alternative. $\lambda$ trades precision against recall.
- **The truth of a GGM learned from a Gaussian Bayesian network is the moral graph** (arrows without heads + married
  co-parents). On SIMULATED signalling cells with a known network, neighbourhood selection and partial-correlation tests
  recovered it almost perfectly, while the graphical lasso added spurious edges between strongly correlated molecules —
  check estimators on simulated data that resemble yours.
- **An edge is a partial correlation** among the *measured* variables — a hypothesis, not proof of regulation, and
  without direction: hiding one molecule created spurious edges among its neighbours, and stable edges can still be
  false.

---
## ✅ Checkpoint

1. In the 5 × 5 example of Section 5.3, which single entry shows that a non-zero covariance does not imply an edge? Explain in one sentence.
   <details><summary>Answer</summary>$\Sigma_{23}\approx0.3$ while $\Theta_{23}=0$: $y_2$ and $y_3$ are correlated only through their common neighbour $y_1$, so they are conditionally independent given $y_1$ (no edge).</details>
2. Compute by hand the partial correlation of $y_1$ and $y_3$ in the 5 × 5 example ($\Theta_{11}=1.3$, $\Theta_{33}=1.2$, $\Theta_{13}=0.8$).
   <details><summary>Answer</summary>$-0.8/\sqrt{1.3\times1.2}=-0.8/1.249=-0.641$: a fairly strong *negative* direct association (positive $\Theta$ entry, negative partial correlation).</details>
3. For $\Theta=\begin{bmatrix}2&-1&0\\-1&2&-1\\0&-1&2\end{bmatrix}$ and zero mean, write down $p(x_1\mid x_2,x_3)$.
   <details><summary>Answer</summary>$x_1\mid x_2,x_3\sim\mathcal N(-\tfrac{-1}{2}x_2-\tfrac{0}{2}x_3,\;\tfrac12)=\mathcal N(0.5\,x_2,\,0.5)$. $x_3$ gets coefficient 0 because it is not a neighbour of $x_1$ (chain $x_1-x_2-x_3$).</details>
4. What is the Markov blanket of $x_5$ in the 8-node ring of Section 5? How would the answer differ if the graph were a directed chain $x_4\to x_5\to x_6$ inside a Bayesian network?
   <details><summary>Answer</summary>Its neighbours $\{x_4,x_6\}$. In a Bayesian network the blanket is parents, children and the children's other parents; for $x_4\to x_5\to x_6$ with no other parents of $x_6$ it is also $\{x_4,x_6\}$, but if $x_6$ had another parent $z$, then $z$ would join the BN blanket — undirected blankets never contain co-parents.</details>
5. A study measures 20,000 genes in 300 tumours. Why can the MLE $\hat\Theta=S^{-1}$ not be used, and what fixes it?
   <details><summary>Answer</summary>After centring, $S$ has rank at most 299 < 20,000, so it is singular and has no inverse (and even with an inverse it would be dense and noisy). The graphical lasso (or neighbourhood selection) adds an $L_1$ penalty; the penalised problem has a unique sparse solution even when $N<K$.</details>
6. Code reading: in `lasso_cd`, what does `partial = b[j] - A[j] @ beta + A[j, j] * beta[j]` compute, and why is `A[j, j] * beta[j]` added back?
   <details><summary>Answer</summary>$b_j-\sum_{k\neq j}A_{jk}\beta_k$: the part of $b_j$ not explained by the *other* coefficients. `A[j] @ beta` includes the term $A_{jj}\beta_j$, which must be removed because coordinate $j$ is the one being re-optimised; soft-thresholding this quantity and dividing by $A_{jj}$ gives the new $\beta_j$.</details>
7. A graphical lasso on real cytometry data finds an edge pkc–jnk with a negative partial correlation. Can you conclude that PKC inhibits JNK?
   <details><summary>Answer</summary>No. The edge says the two are negatively associated after adjusting for the other measured molecules. It gives no direction, could reflect an unmeasured common cause or a mixture of cell states, and assumes linear-Gaussian relations. Establishing inhibition requires an intervention (e.g. a PKC inhibitor or activator) and mechanism.</details>
8. A classmate writes the log-likelihood as $\tfrac12\log\det\Theta-\tfrac12\operatorname{tr}(S\Theta)$ with $S=\frac{1}{2N}\sum_i\mathbf x^i(\mathbf x^i)^\top$. What goes wrong, and what is correct?
   <details><summary>Answer</summary>$\frac{1}{2N}\sum_i(\mathbf x^i)^\top\Theta\mathbf x^i=\tfrac12\operatorname{tr}(S\Theta)$ requires $S=\frac1N\sum_i\mathbf x^i(\mathbf x^i)^\top$. With the $\frac{1}{2N}$ definition the maximiser would be $(S/2)^{-1}=2S^{-1}$, twice the correct precision matrix.</details>
9. The simulated signalling cells were generated from 20 arrows, yet the GGM's true graph has an edge pip3–erk with no arrow between them. Why, and which other pair is in the same situation?
   <details><summary>Answer</summary>pip3 and erk are both parents of akt. In a linear-Gaussian Bayesian network $\Theta=(I-B)^\top\Omega^{-1}(I-B)$, and $\Theta_{ij}$ picks up a term $w_{ic}w_{jc}/\omega_c$ for every shared child $c$: conditioning on akt makes its parents dependent (explaining away). The undirected graph is the *moral graph*; pip3–pka (also parents of akt) is the other married pair.</details>
10. In Section 10.7, hiding pka created an edge p38–jnk. Is the GGM wrong?
   <details><summary>Answer</summary>No — among the 10 measured molecules, p38 and jnk really are partially correlated, because both respond to the unmeasured pka (a latent confounder); marginalising a Gaussian connects all neighbours of the removed variable. The *interpretation* as a direct interaction is wrong. Only measuring pka, or perturbing p38 or jnk, can tell.</details>

## 🧠 Questions & discussion

1. A GGM assumes all samples come from one Gaussian. Single-cell data often contain several cell types or states. Sketch (or simulate) how two cell populations with *no* within-population edge between two genes can still produce a strong partial correlation when pooled. How would you guard against this?
2. Undirected edges avoid the direction problem of Bayesian networks. Is that an advantage or merely postponing the question? For which biological questions is an undirected network enough?
3. The graphical lasso's $\lambda$ trades precision against recall. In a follow-up experiment budget of 10 validation experiments, which $\lambda$ (or which stability threshold) would you choose, and why?
4. Partial correlations condition on *all* other measured variables. What happens to an edge when you add more measured molecules to the panel — can edges appear, disappear, or both?
5. Flow-cytometry intensities are skewed and floored at 0. Which transformation would you try before fitting a GGM, and how would you check whether it helped?

## 🚀 Bonus — going further

- **Choose $\lambda$ by cross-validation.** Use `sklearn.covariance.GraphicalLassoCV` on the simulated signalling cells and compare the selected network with the true moral graph and with the stability results of 10.8. Does maximising held-out likelihood pick a sparse network?
- **Gaussian copula / nonparanormal.** Replace each protein by the normal scores of its ranks (`stats.norm.ppf((rank − 0.5)/N)`) before computing $S$, a simple way to relax the Gaussian assumption; does the network change? (Make the simulated data non-Gaussian first, e.g. `np.exp(D)` — a monotone transformation that skews every molecule — and compare `S⁻¹` before and after the rank transformation.)
- **A directed view.** Compare the undirected GGM skeleton with the Chow–Liu tree and the hill-climbing Bayesian network of [Module 17](17_bayesian_networks_inference_learning.ipynb) (both learned from simulated cells of the same network): which edges do all methods agree on, and which co-parent edges appear only in the GGM?
- **Lasso in genomics.** The [RegularizedLinearRegression](../../RegularizedLinearRegression/) tutorials apply ridge and lasso to high-dimensional wheat genomic data — the same $L_1$ machinery that neighbourhood selection uses.
- **Read the original.** Friedman, Hastie & Tibshirani (2008) and ESL §17.3 analyse the full real Sachs data set (all 7,466 cells) with the graphical lasso; draw the corresponding path plot (number of edges versus $\lambda$) for the 7,466 simulated cells with `graphical_lasso_bcd`.
- **From Gaussian to discrete undirected models.** [Module 37](37_protein_sequences_language_models.ipynb) learns a *Potts model* — a discrete Markov random field — from protein sequence alignments; its couplings play the role of $\Theta$ and predict residue contacts (direct-coupling analysis), and the pseudo-likelihood used there is the discrete cousin of neighbourhood selection. [Module 41](41_perturbation_causal_inference.ipynb) adds the arrows back with perturbation data.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 18 · Gaussian graphical models
Input: SIMULATED signalling cells shaped like Sachs et al. 2005 (2,603 unperturbed cells × 11 molecules); simulated GGMs
Output: precision / partial-correlation matrices; sparse networks along a λ path; edge precision/recall vs the true graph
Method: multivariate Gaussian; MLE Θ̂ = S⁻¹; graphical lasso (block coordinate descent) and neighbourhood selection
Key equation: Θ_ij = 0 ⇔ x_i ⊥ x_j | rest;  x_i | x_−i ~ N(−Σ_j Θ_ij x_j / Θ_ii, 1/Θ_ii)
Key parameters: penalty λ (and the AND/OR rule for neighbourhood selection); sample size N vs number of variables K
What changed when I modified λ / N / the hub strength: …
One failure mode: an unmeasured molecule (pka hidden) creates stable spurious edges among its neighbours (e.g. p38–jnk)
Connection to the 02-620 topics (which idea of the lecture this illustrates): precision matrix = graph; conditionals are
regressions; MLE Θ̂ = S⁻¹; sparsity with the graphical lasso
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its lecture on Gaussian graphical models.
- Sachs K., Perez O., Pe'er D., Lauffenburger D. A., Nolan G. P. (2005) Causal protein-signaling networks derived from multiparameter single-cell data. *Science* 308:523–529.
- Friedman J., Hastie T., Tibshirani R. (2008) Sparse inverse covariance estimation with the graphical lasso. *Biostatistics* 9:432–441.
- Meinshausen N., Bühlmann P. (2006) High-dimensional graphs and variable selection with the lasso. *Annals of Statistics* 34:1436–1462.
- Mazumder R., Hastie T. (2012) The graphical lasso: new insights and alternatives. *Electronic Journal of Statistics* 6:2125–2149.
- Hsieh C.-J., Sustik M. A., Dhillon I. S., Ravikumar P. (2014) QUIC: quadratic approximation for sparse inverse covariance estimation. *Journal of Machine Learning Research* 15:2911–2947.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — Chapter 17 (undirected graphical models; §17.3 analyses the Sachs data).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 4 (Gaussian models), Chapter 19 (Markov random fields), Chapter 26 (graphical model structure learning).
- Whittaker J. (1990) *Graphical Models in Applied Multivariate Statistics*. Wiley; exam-marks data from Mardia K. V., Kent J. T., Bibby J. M. (1979) *Multivariate Analysis*. Academic Press.
- Tibshirani R. (1996) Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society B* 58:267–288.
- Ravikumar P., Wainwright M. J., Raskutti G., Yu B. (2011) High-dimensional covariance estimation by minimizing ℓ1-penalized log-determinant divergence. *Electronic Journal of Statistics* 5:935–980.
- Fisher R. A. (1924) The distribution of the partial correlation coefficient. *Metron* 3:329–332.

## ➡️ Next

Gaussian graphical models describe *simultaneous* measurements. Many biological data are *sequences* —
DNA, protein chains, time courses — in which a hidden state changes along the sequence. Continue with
[Module 19 · Hidden Markov models I: decoding hidden states in sequences](19_hmm_inference.ipynb).